# Lesion-segmentation benchmark — LINDA · SynthStroke · OpenADS

Standalone notebook that compares three lesion-segmentation tools on a
pilot of chronic and acute strokes.

| Dataset                                | Modality              | LINDA | SynthStroke | OpenADS |
|----------------------------------------|-----------------------|:-----:|:-----------:|:-------:|
| ds004884 — Aphasia Recovery | T1w (chronic) | ✔     | ✔           | —       |
| ISLES 2022 (Zenodo 7153326)           | DWI (acute)           | ⚠     | —           | ✔       |
| ds006533 — Duricy 2025 (OpenNeuro)    | T1w (chronic)         | ✔     | ✔           | —       |
| ISLES 2015 SISS (Zenodo 19135955)     | DWI+T1+FLAIR (acute)  | ✔     | ✔           | ✔       |

OpenADS is DWI-based and is auto-skipped on chronic data. SynthStroke is T1w-only and is auto-skipped on acute data (no T1w in ISLES-2022). LINDA was trained on chronic strokes; running it on ISLES is informative as a transfer-fail baseline rather than a fair comparison. ds006533 (Duricy et al., *Cortex* 2025) is a 105-subject chronic aphasia dataset with manual T1w lesion masks.

**The notebook is designed to run top-to-bottom with no prior setup.** It
will:

1. Install missing Python packages.
2. Load Neurodesk modules for the segmentation tools (gracefully skipping
   any that aren't on this system).
3. Clone ds004884 via DataLad and download + unzip ISLES-2022 from
   Zenodo if either is missing.
4. For the DataLad-managed chronic set, run `datalad get` selectively
   on the sampled subjects (no full payload download). ISLES files are
   plain NIfTIs after unzip — no per-subject fetch needed.
5. Run each tool on each eligible subject, with idempotent skip-if-done
   caching.
6. Compute Dice, IoU, volume diff, HD95 + mean surface distance, and
   per-region atlas overlap. Output as CSV + HTML report.


## 1 · Config

In [1]:
# ============================================================
# Knobs you'll touch most often
# ============================================================
CONFIG = {
    # ── Sample size ───────────────────────────────────────────
    "N_PER_DATASET":       2,      # subjects sampled per active dataset
    "RANDOM_SEED":        31,
    "SUBJECT_FILTER":      None,   # ["sub-XXXX", ...] to force-select, else None

    # ── Tool toggles ──────────────────────────────────────────
    "RUN_LINDA":           True,
    "RUN_SYNTHSTROKE":     True,
    "RUN_OPENADS":         True,

    # ── Pipeline behaviour ────────────────────────────────────
    "SKIP_IF_DONE":        True,   # idempotent runners
    "STOP_ON_TOOL_ERROR":  False,  # one tool failing won't kill the run

    # ── Setup behaviour ───────────────────────────────────────
    "AUTO_INSTALL_PIP":    True,   # pip install missing packages
    "AUTO_INSTALL_DATALAD": True,  # pip install datalad if missing
    "AUTO_CLONE_DATASETS": True,   # datalad install if missing
    "AUTO_FETCH_DATA":     True,   # datalad get for sampled subjects

    # ── Metric behaviour ──────────────────────────────────────
    "ATLAS":               "cort-maxprob-thr25-2mm",
    "HD95":                True,

    # ── Output ────────────────────────────────────────────────
    "REPORTS_SUBDIR":      "reports/benchmarks",
    "DERIV_SUBDIR":        "derivatives",
}

# ── Dataset registry ─────────────────────────────────────────────────
# All supported datasets. To add a new one: add an entry here.
#   phase    : "chronic" | "acute"
#   modality : "T1w" | "DWI"  — controls which tools are auto-applied
#   type     : "datalad" (OpenNeuro git clone) | "zenodo" (zip download)
# Tool assignment by phase (unless a tool explicitly skips):
#   chronic → LINDA + SynthStroke
#   acute   → OpenADS + LINDA (transfer-fail baseline)
DATASET_REGISTRY = {
    "ds004884": {
        "phase":    "chronic",
        "label":    "ds004884 — Aphasia Recovery",
        "modality": "T1w",
        "type":     "datalad",
        "url":      "https://github.com/OpenNeuroDatasets/ds004884.git",
        "local":    "data/ds004884",
    },
    "isles2022": {
        "phase":    "acute",
        "label":    "ISLES 2022",
        "modality": "DWI",
        "type":     "zenodo",
        "url":      "https://zenodo.org/records/7153326/files/ISLES-2022.zip?download=1",
        "md5":      "302ee280373cdd5c190ab763d72a7a50",
        "local":    "data/ISLES-2022",
    },
    "ds006533": {
        "phase":    "chronic",
        "label":    "ds006533 — Duricy 2025 (Numeracy/Aphasia)",
        "modality": "T1w",
        "type":     "datalad",
        "url":      "https://github.com/OpenNeuroDatasets/ds006533.git",
        "local":    "data/ds006533",
    },
    "isles2015": {
        "phase":    "acute",
        "label":    "ISLES 2015 SISS",
        "modality": "DWI",       # also has T1w/FLAIR/T2 — all tools can run
        "type":     "zenodo_bids",  # zenodo zip + BIDS conversion after extract
        "url":      "https://zenodo.org/api/records/19135955/files/ISLES2015.zip/content",
        "md5":      None,
        "local":    "data/ISLES2015",
    },
    "synthetic_acute": {
        "phase":       "acute",
        "label":       "Synthetic acute stroke (DWI+ADC)",
        "modality":    "DWI",
        "type":        "synthetic",   # generated locally — no download
        "url":         None,
        "local":       "data/synthetic_acute",
        "n_synth":     6,             # subjects to generate
        "synth_seed":  7,             # RNG seed (reproducible)
        "synth_phase": "acute",       # DWI-bright / ADC-dark / subtle T1
    },
    "synthetic_chronic": {
        "phase":       "chronic",
        "label":       "Synthetic chronic stroke (T1w cavity)",
        "modality":    "T1w",
        "type":        "synthetic",
        "url":         None,
        "local":       "data/synthetic_chronic",
        "n_synth":     6,
        "synth_seed":  17,            # different seed → different cohort
        "synth_phase": "chronic",     # T1 dark cavity / ADC elevated / DWI iso
    },
}

# ── Active datasets ───────────────────────────────────────────────────
# Select which datasets to include in this run.
# Keys must exist in DATASET_REGISTRY above.
# Comment out any line to exclude a dataset.
ACTIVE_DATASETS = [
    "ds004884",   # chronic, T1w  → LINDA + SynthStroke
    "isles2022",  # acute,   DWI  → OpenADS + LINDA (transfer baseline) — ~1.7 GB Zenodo download on first run
    "ds006533",   # chronic, T1w  → LINDA + SynthStroke
    "isles2015",  # acute,   DWI  → OpenADS + LINDA (transfer baseline)
    "synthetic_acute",    # acute,   synthetic → OpenADS (+ LINDA/SynthStroke transfer baseline)
    "synthetic_chronic",  # chronic, synthetic → LINDA + SynthStroke
]

# ── Resolve paths ─────────────────────────────────────────────────────
from pathlib import Path
import os

PROJECT_DIR = Path(globals().get("PROJECT_DIR", os.getcwd())).resolve()
REPORTS_DIR = PROJECT_DIR / CONFIG["REPORTS_SUBDIR"]
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR  {PROJECT_DIR}")
print(f"REPORTS_DIR  {REPORTS_DIR}")
print()
for ds_key in ACTIVE_DATASETS:
    ds   = DATASET_REGISTRY[ds_key]
    root = PROJECT_DIR / ds["local"]
    status = "✔ exists" if root.exists() else "✘ NOT YET CLONED"
    print(f"  {ds_key:12s}  [{ds['phase']:7s} / {ds['modality']:3s}]  {status}")


PROJECT_DIR  /home/jovyan/neurodesktop-storage/calmar
REPORTS_DIR  /home/jovyan/neurodesktop-storage/calmar/reports/benchmarks

  ds004884      [chronic / T1w]  ✔ exists
  isles2022     [acute   / DWI]  ✔ exists
  ds006533      [chronic / T1w]  ✔ exists
  isles2015     [acute   / DWI]  ✔ exists
  synthetic_acute  [acute   / DWI]  ✘ NOT YET CLONED
  synthetic_chronic  [chronic / T1w]  ✘ NOT YET CLONED


## 2 · Bootstrap — Python packages
Installs `nibabel`, `nilearn`, `scipy`, `pandas`, `datalad` if any are missing. Set `CONFIG['AUTO_INSTALL_PIP'] = False` to disable.

In [2]:
import importlib, subprocess, sys

REQUIRED = {
    "nibabel":  "nibabel",
    "nilearn":  "nilearn",
    "scipy":    "scipy",
    "pandas":   "pandas",
    "numpy":    "numpy",
}

def _import_or_pip(pkg_name: str, pip_name: str) -> bool:
    try:
        importlib.import_module(pkg_name)
        return True
    except ImportError:
        if not CONFIG["AUTO_INSTALL_PIP"]:
            print(f"  ✘ {pkg_name} missing (AUTO_INSTALL_PIP=False)")
            return False
        print(f"  installing {pip_name} …")
        cmd = [sys.executable, "-m", "pip", "install", "-q", pip_name]
        # `--break-system-packages` is needed on Neurodesk's PEP668 envs
        try:
            subprocess.check_call(cmd + ["--break-system-packages"])
        except subprocess.CalledProcessError:
            subprocess.check_call(cmd)
        importlib.import_module(pkg_name)
        return True

print("Python deps:")
all_ok = True
for pkg, pip in REQUIRED.items():
    ok = _import_or_pip(pkg, pip)
    print(f"  {'✔' if ok else '✘'}  {pkg}")
    all_ok &= ok

# DataLad is bigger; only install if we'll need to clone or fetch.
if CONFIG["AUTO_INSTALL_DATALAD"] and (CONFIG["AUTO_CLONE_DATASETS"] or CONFIG["AUTO_FETCH_DATA"]):
    try:
        importlib.import_module("datalad")
        print("  ✔  datalad")
    except ImportError:
        print("  installing datalad (~1 min) …")
        cmd = [sys.executable, "-m", "pip", "install", "-q", "datalad"]
        try:
            subprocess.check_call(cmd + ["--break-system-packages"])
        except subprocess.CalledProcessError:
            subprocess.check_call(cmd)
        print("  ✔  datalad")


Python deps:
  ✔  nibabel
  ✔  nilearn
  ✔  scipy
  ✔  pandas
  ✔  numpy
  ✔  datalad


## 3 · Bootstrap — Neurodesk modules

In a Neurodesk kernel `module.load(...)` makes external tools (LINDA, SynthStroke, OpenADS, FSL, ANTs, etc.) available on `PATH`. Outside Neurodesk, `module` is undefined — the cell prints a warning and continues. Adjust the version pins to whatever `ml av <tool>` shows on your system.

In [3]:
import shutil, module

# Single batched module.load — same form as the original linda notebook.
# Edit the version pins to match `ml av <tool>` output on your Neurodesk
# system. If `module` isn't injected (running outside Neurodesk), the
# try/except prints a warning and the rest of the notebook still runs.

if "module" in globals():
    try:
        await module.load('linda/0.5.1', 'hdbet/1.0.0', 'fsl/6.0.7.22', 'ants/2.6.5', 'synthstroke/0.0.0', 'openadscpu/1.0.0', 'freesurfer/7.4.1')
        await module.list()
    except Exception as exc:
        print(f"⚠ module.load failed: {type(exc).__name__}: {exc}")
        print("  Check the pins above against `ml av <tool>` and re-run.")
else:
    print("⚠ Neurodesk `module` API not available in this kernel.")
    print("  Tools that need a Neurodesk module won't be on PATH and their")
    print("  runners will skip with a clear error.")

# CLI availability check after the load, grouped by the tool that
# provides each binary. (Module names like `openadscpu` aren't binaries —
# they put a binary like `ads` on PATH. Checking the module name itself
# would always report ✘.)
print()
print("CLIs on PATH after module loads:")
CLI_GROUPS = [
    ("LINDA",         ["linda_predict.sh"]),
    ("HD-BET",        ["hd-bet"]),
    ("FSL",           ["fslmaths", "mri_synthstrip"]),
    ("Freesurfer",    ["mri_synthstrip"]),
    ("ANTs",          ["antsRegistrationSyN.sh", "antsApplyTransforms"]),
    ("SynthStroke",   ["synthstroke"]),
    ("OpenADS",       ["ads"]),           # module is openadscpu, binary is `ads`
    ("DataLad/annex", ["datalad", "git-annex"]),
    ("Container",     ["singularity", "apptainer"]),
]
for group, names in CLI_GROUPS:
    found = [(n, shutil.which(n)) for n in names]
    any_ok = any(p for _, p in found)
    print(f"  {'✔' if any_ok else '✘'}  {group}")
    for n, p in found:
        print(f"      {'✔' if p else '✘'} {n:25s} {p or ''}")



CLIs on PATH after module loads:
  ✔  LINDA
      ✔ linda_predict.sh          /cvmfs/neurodesk.ardc.edu.au/containers/linda_0.5.1_20260508/linda_predict.sh
  ✔  HD-BET
      ✔ hd-bet                    /cvmfs/neurodesk.ardc.edu.au/containers/hdbet_1.0.0_20210318/hd-bet
  ✔  FSL
      ✔ fslmaths                  /cvmfs/neurodesk.ardc.edu.au/containers/fsl_6.0.7.22_20260416/fslmaths
      ✔ mri_synthstrip            /cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_7.4.1_20231214/mri_synthstrip
  ✔  Freesurfer
      ✔ mri_synthstrip            /cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_7.4.1_20231214/mri_synthstrip
  ✔  ANTs
      ✔ antsRegistrationSyN.sh    /cvmfs/neurodesk.ardc.edu.au/containers/ants_2.6.5_20260602/antsRegistrationSyN.sh
      ✔ antsApplyTransforms       /cvmfs/neurodesk.ardc.edu.au/containers/ants_2.6.5_20260602/antsApplyTransforms
  ✔  SynthStroke
      ✔ synthstroke               /cvmfs/neurodesk.ardc.edu.au/containers/synthstroke_0.0.0_20260518/synthstrok

## 4 · Bootstrap — Install datasets if missing

Two different mechanisms because the datasets live in different places:

- **Chronic — ds004884 (DataLad/OpenNeuro).** `datalad install` is metadata-only — clones the dataset structure and lesion pointer files but doesn't download the multi-gigabyte NIfTI payload. We fetch payload selectively in step 6 once we know which subjects we want.
- **Acute — ISLES 2022 (Zenodo).** Single ~1.7 GB zip with the BIDS-style dataset baked in. The cell downloads it once (md5-verified), extracts it into `PROJECT_DIR/ISLES-2022/`, then skips on subsequent runs.

In [4]:
import hashlib, urllib.request, zipfile
import sys as _sys

# Synthetic acute-stroke generator lives in its own module (testable +
# importable). PROJECT_DIR is defined in the config cell above.
if str(PROJECT_DIR) not in _sys.path:
    _sys.path.insert(0, str(PROJECT_DIR))
from synthetic_stroke import synthetic_install

def datalad_install(url: str, target: Path) -> bool:
    """Clone an OpenNeuro DataLad dataset (metadata only)."""
    if target.exists():
        print(f"  ✔  {target.name} already cloned")
        return True
    if not CONFIG["AUTO_CLONE_DATASETS"]:
        print(f"  ✘  {target.name} missing (AUTO_CLONE_DATASETS=False)")
        return False
    if not shutil.which("datalad"):
        print(f"  ✘  datalad CLI missing — can't clone {target.name}")
        return False
    print(f"  cloning {target.name} from {url} …")
    proc = subprocess.run(["datalad", "install", "-s", url, str(target)],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  ✘  datalad install failed: {proc.stderr.strip()[:300]}")
        return False
    print(f"  ✔  {target.name} cloned")
    return True


def _md5(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()


def _looks_extracted(target: Path) -> bool:
    """ISLES is considered "installed" if at least one sub-strokecase
    directory exists under target/ (or target/ISLES-2022/, if the zip
    nested itself one level deep)."""
    if not target.exists():
        return False
    for root in (target, target / target.name):
        if any(root.glob("sub-strokecase*")):
            return True
    return False


def zenodo_install(url: str, target: Path, *, md5: str | None = None) -> bool:
    """Download + extract a Zenodo zip. Idempotent: returns True if the
    extracted dataset already exists, or if download+unzip succeeded."""
    if _looks_extracted(target):
        print(f"  ✔  {target.name} already extracted")
        return True
    if not CONFIG["AUTO_CLONE_DATASETS"]:
        print(f"  ✘  {target.name} missing (AUTO_CLONE_DATASETS=False)")
        return False

    target.parent.mkdir(parents=True, exist_ok=True)
    zip_path = target.parent / f"{target.name}.zip"

    # Download if missing (or if md5 mismatches an existing partial file)
    need_dl = True
    if zip_path.exists() and md5:
        print(f"  found existing {zip_path.name}, checking md5 …")
        if _md5(zip_path) == md5:
            need_dl = False
            print(f"  ✔  md5 matches, skipping download")
        else:
            print(f"  ✘  md5 mismatch, re-downloading")
            zip_path.unlink()

    if need_dl:
        print(f"  downloading {target.name} from Zenodo (~1.7 GB) — "
              f"this can take several minutes …")
        try:
            urllib.request.urlretrieve(url, zip_path)
        except Exception as exc:
            print(f"  ✘  download failed: {type(exc).__name__}: {exc}")
            return False
        if md5 and _md5(zip_path) != md5:
            print(f"  ✘  md5 mismatch after download — file may be corrupt")
            return False
        print(f"  ✔  download complete ({zip_path.stat().st_size:,} bytes)")

    print(f"  extracting → {target} …")
    try:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target.parent)
    except Exception as exc:
        print(f"  ✘  extract failed: {type(exc).__name__}: {exc}")
        return False

    # Some Zenodo zips wrap content in a top-level folder of the same name.
    # If the expected sub-strokecase dirs are one level deep, lift them up.
    if not list(target.glob("sub-strokecase*")):
        nested = target / target.name
        if nested.exists() and list(nested.glob("sub-strokecase*")):
            for child in nested.iterdir():
                child.rename(target / child.name)
            nested.rmdir()

    print(f"  ✔  {target.name} ready "
          f"({len(list(target.glob('sub-strokecase*')))} subjects)")
    return True



# ── ISLES 2015 SISS: VSD → BIDS converter ────────────────────────────
# The Zenodo zip extracts to:
#   ISLES2015/ISLES2015_SISS/training/<n>/
#     VSD.Brain.XX.O.MR_DWI.<id>/    ← NIfTI file(s) inside
#     VSD.Brain.XX.O.MR_T1.<id>/
#     VSD.Brain.XX.O.MR_T2.<id>/
#     VSD.Brain.XX.O.MR_Flair.<id>/
#     VSD.Brain.XX.O.OT.<id>/        ← ground-truth mask
# Converted to a flat BIDS-ish layout compatible with discover():
#   target/sub-001/dwi/sub-001_dwi.nii.gz
#   target/sub-001/anat/sub-001_T1w.nii.gz  etc.
#   target/derivatives/sub-001/sub-001_msk.nii.gz

_VSD_BIDS_MAP = {
    "MR_DWI":   ("dwi",  "{sub}_dwi.nii.gz"),
    "MR_T1":    ("anat", "{sub}_T1w.nii.gz"),
    "MR_T2":    ("anat", "{sub}_T2w.nii.gz"),
    "MR_Flair": ("anat", "{sub}_FLAIR.nii.gz"),
}


def _vsd_key(folder_name: str) -> str | None:
    """Extract modality key from a VSD folder name.
    'VSD.Brain.XX.O.MR_DWI.10280' → 'MR_DWI'
    'VSD.Brain.XX.O.OT.10285'     → 'OT'
    """
    parts = folder_name.split(".")
    return parts[4] if len(parts) >= 5 else None


def bids_convert_isles2015(target: Path) -> int:
    """Convert VSD-format ISLES 2015 SISS to BIDS in-place inside *target*.
    Returns the number of subjects newly converted.
    Idempotent: skips subjects whose sub-NNN/dwi/ already exists.
    """
    import shutil, re, gzip

    training_dir = target / "ISLES2015_SISS" / "training"
    if not training_dir.exists():
        print(f"  ✘  bids_convert: training dir not found at {training_dir}")
        return 0

    subj_dirs = sorted(
        [d for d in training_dir.iterdir()
         if d.is_dir() and re.match(r"^\d+$", d.name)],
        key=lambda d: int(d.name),
    )
    if not subj_dirs:
        print(f"  ✘  bids_convert: no numeric subject dirs under {training_dir}")
        return 0

    converted = 0
    for subj_dir in subj_dirs:
        sub = f"sub-{int(subj_dir.name):03d}"
        # Verify the previously-converted files are real gzips. If a
        # prior run wrote a plain .nii under a .nii.gz name (old bug), wipe
        # those bad files so the loop below recreates them correctly.
        if (target / sub / "dwi").exists():
            bad = []
            for chk in (target / sub).rglob("*.nii.gz"):
                try:
                    with open(chk, "rb") as _ck:
                        if _ck.read(2) != b"\x1f\x8b":
                            bad.append(chk)
                except OSError:
                    bad.append(chk)
            if not bad:
                continue
            print(f"  ↺ {sub}: removing {len(bad)} non-gzip file(s) and re-converting")
            for b in bad:
                b.unlink()

        for vsd_dir in sorted(subj_dir.iterdir()):
            if not vsd_dir.is_dir():
                continue
            key = _vsd_key(vsd_dir.name)
            niis = sorted(vsd_dir.glob("*.nii.gz")) or sorted(vsd_dir.glob("*.nii"))
            if not niis:
                continue
            src_nii = niis[0]

            if key in _VSD_BIDS_MAP:
                folder, fname_tmpl = _VSD_BIDS_MAP[key]
                dst_dir = target / sub / folder
                dst_dir.mkdir(parents=True, exist_ok=True)
                dst = dst_dir / fname_tmpl.format(sub=sub)
                # BIDS-Derivatives mandates *.nii.gz, but ISLES 2015 source
                # files are uncompressed *.nii. shutil.copy2 with the .gz
                # destination silently produced files with the wrong magic;
                # always gzip-compress if the source isn't already gzipped.
                if src_nii.suffix == ".gz":
                    shutil.copy2(src_nii, dst)
                else:
                    with open(src_nii, "rb") as fi, gzip.open(dst, "wb") as fo:
                        shutil.copyfileobj(fi, fo)

            elif key == "OT":
                dst_dir = target / "derivatives" / sub
                dst_dir.mkdir(parents=True, exist_ok=True)
                dst = dst_dir / f"{sub}_msk.nii.gz"
                if src_nii.suffix == ".gz":
                    shutil.copy2(src_nii, dst)
                else:
                    with open(src_nii, "rb") as fi, gzip.open(dst, "wb") as fo:
                        shutil.copyfileobj(fi, fo)

        converted += 1

    print(f"  ✔  bids_convert: converted {converted} new subjects "
          f"({len(subj_dirs)} total in training set)")
    return converted


def _isles2015_bids_done(target: Path) -> bool:
    """True if BIDS conversion already complete AND every *.nii.gz under
    sub-001/ has valid gzip magic. Older runs wrote raw .nii bytes under
    a .nii.gz name; that state must be re-converted, not skipped."""
    root = target / "sub-001"
    if not root.exists():
        return False
    for p in root.rglob("*.nii.gz"):
        try:
            with open(p, "rb") as fh:
                if fh.read(2) != b"\x1f\x8b":
                    return False
        except OSError:
            return False
    return True


def zenodo_bids_install(url: str, target: Path, *,
                        bids_convert_fn, md5: str | None = None) -> bool:
    """Download + extract a Zenodo zip, then run a BIDS converter.
    Fully idempotent: returns True if the BIDS layout already exists.
    """
    if _isles2015_bids_done(target):
        print(f"  ✔  {target.name} already converted to BIDS")
        return True
    if not CONFIG["AUTO_CLONE_DATASETS"]:
        print(f"  ✘  {target.name} missing (AUTO_CLONE_DATASETS=False)")
        return False

    target.parent.mkdir(parents=True, exist_ok=True)
    zip_path = target.parent / f"{target.name}.zip"

    # ── Download ──
    need_dl = True
    if zip_path.exists() and md5:
        print(f"  found existing {zip_path.name}, checking md5 …")
        if _md5(zip_path) == md5:
            need_dl = False
            print(f"  ✔  md5 matches, skipping download")
        else:
            print(f"  ✘  md5 mismatch, re-downloading")
            zip_path.unlink()

    if need_dl:
        print(f"  downloading {target.name} from Zenodo — "
              f"this can take several minutes …")
        try:
            urllib.request.urlretrieve(url, zip_path)
        except Exception as exc:
            print(f"  ✘  download failed: {type(exc).__name__}: {exc}")
            return False
        if md5 and _md5(zip_path) != md5:
            print(f"  ✘  md5 mismatch after download — file may be corrupt")
            return False
        print(f"  ✔  download complete ({zip_path.stat().st_size:,} bytes)")

    # ── Extract ──
    if not target.exists() or not any(target.iterdir()):
        print(f"  extracting → {target} …")
        try:
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(target.parent)
        except Exception as exc:
            print(f"  ✘  extract failed: {type(exc).__name__}: {exc}")
            return False
        print(f"  ✔  extraction complete")
    else:
        print(f"  ✔  already extracted, skipping unzip")

    # ── BIDS conversion ──
    print(f"  running BIDS conversion …")
    bids_convert_fn(target)
    return True


print("Datasets:")
for _ds_key in ACTIVE_DATASETS:
    _ds = DATASET_REGISTRY[_ds_key]
    _target = PROJECT_DIR / _ds["local"]
    if _ds["type"] == "datalad":
        datalad_install(_ds["url"], _target)
    elif _ds["type"] == "zenodo":
        zenodo_install(_ds["url"], _target, md5=_ds.get("md5"))
    elif _ds["type"] == "zenodo_bids":
        zenodo_bids_install(_ds["url"], _target,
                            bids_convert_fn=bids_convert_isles2015,
                            md5=_ds.get("md5"))
    elif _ds["type"] == "synthetic":
        synthetic_install(_target,
                          n_subjects=_ds.get("n_synth", 6),
                          seed=_ds.get("synth_seed", 7),
                          phase=_ds.get("synth_phase", "acute"))
    else:
        print(f"  ✘ Unknown type '{_ds['type']}' for dataset {_ds_key}")


Datasets:
  ✔  ds004884 already cloned
  ✔  ISLES-2022 already extracted
  ✔  ds006533 already cloned
  ✔  ISLES2015 already converted to BIDS
  generating 6 synthetic ACUTE subjects (T1w+DWI+ADC) → /home/jovyan/neurodesktop-storage/calmar/data/synthetic_acute …
    ✔ sub-synacu000  (2,360 lesion voxels)
    ✔ sub-synacu001  (1,156 lesion voxels)
    ✔ sub-synacu002  (1,803 lesion voxels)
    ✔ sub-synacu003  (2,077 lesion voxels)
    ✔ sub-synacu004  (427 lesion voxels)
    ✔ sub-synacu005  (2,155 lesion voxels)
  ✔  synthetic acute dataset ready (6 subjects)
  generating 6 synthetic CHRONIC subjects (T1w+DWI+ADC) → /home/jovyan/neurodesktop-storage/calmar/data/synthetic_chronic …
    ✔ sub-synchr000  (805 lesion voxels)
    ✔ sub-synchr001  (477 lesion voxels)
    ✔ sub-synchr002  (1,929 lesion voxels)
    ✔ sub-synchr003  (2,225 lesion voxels)
    ✔ sub-synchr004  (2,814 lesion voxels)
    ✔ sub-synchr005  (655 lesion voxels)
  ✔  synthetic chronic dataset ready (6 subjects)


## 5 · Helpers — gzip check, annex fetch, metrics

In [5]:
import json, glob, time
import numpy as np, pandas as pd, nibabel as nib
from nilearn.image import resample_to_img
from IPython.display import display, HTML

GZIP_MAGIC = b"\x1f\x8b"

def is_gzip(path: Path) -> bool:
    try:
        with open(path, "rb") as fh:
            return fh.read(2) == GZIP_MAGIC
    except OSError:
        return False


def datalad_get(target: Path, timeout=900, stream: bool = True) -> Path | None:
    """Materialise a single git-annex-managed file.

    Tries, in order: the in-process datalad API, the `datalad` CLI, then
    `git annex get`. Returns the path once the real content is on disk,
    else None.

    Robustness fixes (why auto-fetch previously did nothing):
      * An UNFETCHED annex file is a *broken symlink* -- its target under
        .git/annex/objects/ does not exist yet -- so Path.exists() is False
        even though the pointer is present and fetchable. The old guard
        `if not target.exists(): return None` bailed before trying to fetch.
        We now also accept broken symlinks.
      * `datalad` / `git-annex` may not be on the kernel subprocess PATH
        (pip-installed next to sys.executable). We use the in-process
        datalad API first, then look for the binaries next to the running
        Python, and surface the real error instead of failing silently.
    """
    if target is None:
        return None
    target = Path(target)
    # Truly absent -> nothing to do. A broken symlink (unfetched annex
    # pointer) has is_symlink()==True / exists()==False; keep going.
    if not target.exists() and not target.is_symlink():
        return None
    if target.exists() and is_gzip(target):        # already materialised
        return target

    # Dataset root (dir containing .git)
    repo = target
    while repo != repo.parent and not (repo / ".git").exists():
        repo = repo.parent
    if not (repo / ".git").exists():
        print(f"  [fetch] {target.name}: no .git found above the file")
        return None
    rel = target.relative_to(repo)

    def _materialised():
        return target.exists() and is_gzip(target)

    # 1) In-process datalad API -- independent of the `datalad` console
    #    script being on PATH (still shells out to git-annex internally).
    try:
        import datalad.api as _dl
        print(f"  [fetch] datalad.api.get {rel} (dataset={repo.name}) ...")
        try:
            _dl.get(path=str(rel), dataset=str(repo))
        except Exception as exc:
            print(f"  [fetch] datalad.api.get: {type(exc).__name__}: {str(exc)[:300]}")
        if _materialised():
            print(f"  [fetch] {rel}: ok materialised ({target.stat().st_size:,} B)")
            return target
    except ImportError:
        pass

    # 2) CLI fallback -- locate binaries next to sys.executable too.
    import os
    _bindir = os.path.dirname(sys.executable)
    def _exe(name):
        return (shutil.which(name)
                or next((p for p in (os.path.join(_bindir, name),)
                         if os.path.exists(p)), None))
    for name, args in (("datalad", ["get", str(rel)]),
                       ("git", ["annex", "get", str(rel)])):
        exe = _exe(name)
        if not exe:
            continue
        print(f"  [fetch] {os.path.basename(exe)} {' '.join(args)} (cwd={repo.name}) ...")
        try:
            kw = {} if stream else dict(capture_output=True, text=True)
            proc = subprocess.run([exe] + args, cwd=str(repo), timeout=timeout, **kw)
        except subprocess.TimeoutExpired:
            print(f"  [fetch] {rel}: TIMED OUT after {timeout}s")
            return None
        if proc.returncode == 0 and _materialised():
            print(f"  [fetch] {rel}: ok materialised ({target.stat().st_size:,} B)")
            return target
        _err = "" if stream else (proc.stderr or proc.stdout or "")
        print(f"  [fetch] {rel}: {name} rc={proc.returncode}"
              + (f"  err={_err.strip()[:400]}" if _err else ""))

    if shutil.which("git-annex") is None and not _exe("git"):
        print(f"  [fetch] {rel}: FAILED git-annex not found -- datalad needs it "
              f"to fetch annexed content (install git-annex).")
    else:
        print(f"  [fetch] {rel}: FAILED not materialised -- annex content may be "
              f"unavailable from the remote, or network/auth is blocked.")
    return None


def binarise(img):
    return (img.get_fdata() > 0).astype(np.uint8)


def voxel_volume_mm3(img):
    return float(np.prod(img.header.get_zooms()[:3]))


def volume_ml(mask, vv):
    return int(mask.sum()) * vv / 1000.0


def dice_iou(a, b):
    a = a.astype(bool); b = b.astype(bool)
    inter = int((a & b).sum()); s = int(a.sum()) + int(b.sum())
    union = s - inter
    return ((2 * inter / s) if s else float("nan"),
            (inter / union) if union else float("nan"))


def surface_distances(a, b, zooms):
    """HD95 and mean surface distance in mm."""
    out = {"hd95_mm": float("nan"), "mean_surf_mm": float("nan")}
    if a.sum() == 0 or b.sum() == 0:
        return out
    try:
        from scipy.ndimage import binary_erosion, distance_transform_edt
    except ImportError:
        return out
    def boundary(m):
        return m & ~binary_erosion(m, iterations=1)
    ba = boundary(a.astype(bool)); bb = boundary(b.astype(bool))
    if ba.sum() == 0 or bb.sum() == 0:
        return out
    dt_b = distance_transform_edt(~bb, sampling=zooms)
    dt_a = distance_transform_edt(~ba, sampling=zooms)
    d = np.concatenate([dt_b[ba], dt_a[bb]])
    out["hd95_mm"]      = float(np.percentile(d, 95))
    out["mean_surf_mm"] = float(d.mean())
    return out

print("Helpers loaded ✔")


Helpers loaded ✔


## 6 · Discover subjects and fetch their data

Walks each dataset, samples `N` subjects per dataset (seed-stable), then runs `datalad get` on the T1w / DWI / lesion-mask files for just those subjects — keeping the download proportional to the pilot size, not the full dataset.

In [6]:
import random

def _glob_first(base: Path, *patterns) -> Path | None:
    if not base.exists():
        return None
    for pat in patterns:
        hits = sorted(base.glob(pat))
        if hits:
            return hits[0]
    return None


def _t1w(ds: Path, sub: str, ses: str | None) -> Path | None:
    base = (ds / sub / ses / "anat") if ses else (ds / sub / "anat")
    return _glob_first(base, "*T1w.nii*", "*T1w*.nii*")


def _dwi(ds: Path, sub: str, ses: str | None) -> Path | None:
    base = (ds / sub / ses / "dwi") if ses else (ds / sub / "dwi")
    return _glob_first(base, "*dwi.nii*", "*DWI*.nii*")


def discover(ds_root: Path, *, need_t1w=False, need_dwi=False) -> list[dict]:
    out = []
    if not ds_root.exists():
        return out
    for sub_dir in sorted(p for p in ds_root.iterdir()
                          if p.is_dir() and p.name.startswith("sub-")):
        sub = sub_dir.name
        sessions = [p.name for p in sorted(sub_dir.iterdir())
                    if p.is_dir() and p.name.startswith("ses-")] or [None]
        for ses in sessions:
            e = {"subject": sub, "session": ses,
                 "t1w": _t1w(ds_root, sub, ses),
                 "dwi": _dwi(ds_root, sub, ses)}
            if need_t1w and not e["t1w"]: continue
            if need_dwi and not (e["dwi"] or e["t1w"]): continue
            out.append(e)
    return out


def sample(entries: list[dict], n: int) -> list[dict]:
    if not entries:
        return []
    if CONFIG["SUBJECT_FILTER"]:
        entries = [e for e in entries if e["subject"] in CONFIG["SUBJECT_FILTER"]]
    if n is None or n >= len(entries):
        return entries
    rng = random.Random(CONFIG["RANDOM_SEED"])
    pick = rng.sample(entries, n)
    pick.sort(key=lambda e: (e["subject"], e["session"] or ""))
    return pick



def _all_lesion_pointers(ds_root: Path, sub: str, ses: str) -> list[Path]:
    """Find every existing lesion-mask path for this subject across the
    BIDS derivatives layouts the supported datasets use:

    - ds004884:   derivatives/lesion_masks/<sub>/<ses>/anat/*desc-lesion_mask*
    - ISLES 2022: derivatives/<sub>/<ses>/<sub>_<ses>_msk.nii.gz
                  (a single per-subject mask, not nested under anat/)
    """
    candidates: list[Path] = []

    # ds004884-style trees and other common variants
    masked_roots = [
        ds_root / "derivatives" / "lesion_masks" / sub,
        ds_root / "derivatives" / "lesion_masks_native" / sub,
        ds_root / "derivatives" / "lesion_masks_mni" / sub,
        ds_root / "derivatives" / "manual_lesion" / sub,
    ]
    for root in masked_roots:
        if not root.exists():
            continue
        for pattern in ([f"{ses}/anat/*lesion*.nii*", f"{ses}/*lesion*.nii*"]
                        if ses else []) + ["ses-*/anat/*lesion*.nii*",
                                            "ses-*/*lesion*.nii*",
                                            "anat/*lesion*.nii*",
                                            "*lesion*.nii*"]:
            candidates.extend(root.glob(pattern))

    # ISLES 2022 layout: derivatives/<sub>/<ses>/<sub>_<ses>_msk.nii.gz
    # (no anat/ subdir, no "lesion" substring in the filename)
    isles_root = ds_root / "derivatives" / sub
    if isles_root.exists():
        for pattern in ([f"{ses}/*msk*.nii*"] if ses else []) + [
                "ses-*/*msk*.nii*", "*msk*.nii*"]:
            candidates.extend(isles_root.glob(pattern))

    return sorted(set(candidates))


def fetch_for(entry: dict, ds_root: Path) -> dict:
    """Pre-fetch payload for one subject: T1w, DWI, every expert mask.
    Returns a dict of {label: True/False/None} so the caller can summarise."""
    out = {"t1w": None, "dwi": None, "mask": None, "mask_paths": []}
    if not CONFIG["AUTO_FETCH_DATA"]:
        return out
    sub, ses = entry["subject"], entry["session"] or ""

    # Report what is ACTUALLY usable on disk, not whether datalad_get
    # "succeeded". datalad_get returns None for several non-fatal reasons
    # (content already materialised by another process, no special remote,
    # datalad/git-annex not on PATH) — so keying the flags off its return
    # value falsely reported T1w/mask as absent even when a real, readable
    # NIfTI was sitting on disk. We attempt a fetch (in case it's still an
    # annex pointer), then check the file itself.
    def _usable(p):
        p = Path(p)
        return p.exists() and is_gzip(p)

    for key in ("t1w", "dwi"):
        p = entry.get(key)
        if p is None:
            continue
        datalad_get(p)            # materialise if it's an unfetched annex pointer
        out[key] = _usable(p)     # ← actual on-disk state

    masks = _all_lesion_pointers(ds_root, sub, ses)
    out["mask_paths"] = [str(p) for p in masks]
    if not masks:
        print(f"  [fetch] {sub}/{ses}: no lesion mask file found under "
              f"derivatives/lesion_mask* — discoverable layouts:")
        for guess in (ds_root / "derivatives").glob("lesion_*"):
            print(f"      • {guess.relative_to(ds_root)}")
        out["mask"] = False
    else:
        ok = []
        for p in masks:
            datalad_get(p)        # materialise pointer if needed
            ok.append(_usable(p)) # ← actual on-disk state
        out["mask"] = any(ok)
    return out


def _bids_layout_hint(ds_root: Path, n: int = 3) -> str:
    """Tiny diagnostic: list a few subjects and their immediate children
    so the user can confirm the BIDS layout they actually have on disk."""
    lines = []
    if not ds_root.exists():
        return f"  (dataset root does not exist: {ds_root})"
    subs = sorted(p for p in ds_root.iterdir()
                  if p.is_dir() and p.name.startswith("sub-"))[:n]
    if not subs:
        return f"  (no sub-* directories under {ds_root})"
    for s in subs:
        children = sorted(c.name for c in s.iterdir())[:6]
        lines.append(f"  {s.name}/  →  {', '.join(children)}")
    if len(subs) == n:
        lines.append(f"  … and more")
    return "\n".join(lines)


# ── Discover subjects for each active dataset ──────────────────────────
# BENCHMARK_DATASETS: ds_key → {root, subjects, phase, label, run_label, modality}
BENCHMARK_DATASETS = {}

for _ds_key in ACTIVE_DATASETS:
    _ds_meta   = DATASET_REGISTRY[_ds_key]
    _root      = PROJECT_DIR / _ds_meta["local"]
    _phase     = _ds_meta["phase"]
    _modality  = _ds_meta["modality"]
    _run_label = f"{_phase}_{_ds_key}"

    need_t1w = (_modality == "T1w")
    need_dwi = (_modality == "DWI")

    print(f"\n=== {_ds_meta['label']} — discovering subjects ===")
    print(_bids_layout_hint(_root))
    _all = discover(_root, need_t1w=need_t1w, need_dwi=need_dwi)
    _selected = sample(_all, CONFIG["N_PER_DATASET"])
    print(f"  → {len(_all)} subjects with {_modality}, sampling {len(_selected)}")

    BENCHMARK_DATASETS[_ds_key] = {
        "root":       _root,
        "subjects":   _selected,
        "phase":      _phase,
        "label":      _ds_meta["label"],
        "run_label":  _run_label,
        "modality":   _modality,
    }

# ── ISLES-specific warning ─────────────────────────────────────────────
_isles = BENCHMARK_DATASETS.get("isles2022", {})
if _isles and not _isles["subjects"] and _isles["root"].exists():
    print()
    print("⚠ ISLES 2022 is on disk but no DWI-bearing subjects matched.")
    print("  Inspect the layout above — ISLES uses `sub-strokecase####`")
    print("  with `<sub>/<ses>/dwi/<sub>_<ses>_dwi.nii.gz`. If your")
    print("  extract sits one directory deeper (e.g. ISLES-2022/ISLES-2022/),")
    print("  re-run the install cell — it tries to lift nested content.")

# ── Fetch payload ──────────────────────────────────────────────────────
fetch_summary = []
if CONFIG["AUTO_FETCH_DATA"]:
    for _ds_key, _ds_info in BENCHMARK_DATASETS.items():
        _entries   = _ds_info["subjects"]
        _root      = _ds_info["root"]
        _run_label = _ds_info["run_label"]
        if not _entries:
            continue
        print(f"\n=== Fetching payload for {len(_entries)} {_run_label} subject(s) ===")
        for e in _entries:
            sub, ses = e["subject"], e["session"] or ""
            print(f"\n• {sub} {ses}")
            res = fetch_for(e, _root)
            fetch_summary.append({
                "dataset":  _run_label,
                "subject":  sub,
                "session":  ses,
                **{k: v for k, v in res.items() if k != "mask_paths"},
                "n_masks":  len(res["mask_paths"]),
            })

# ── Fetch summary ──────────────────────────────────────────────────────
if fetch_summary:
    print()
    print("=== Fetch summary ===")
    fs_df = pd.DataFrame(fetch_summary)
    display(fs_df)
    for _ds_key, _ds_info in BENCHMARK_DATASETS.items():
        _run_label = _ds_info["run_label"]
        _rows = fs_df[fs_df["dataset"] == _run_label]
        if _rows.empty:
            continue
        _n = len(_rows)
        _n_mask = int((_rows["mask"] == True).sum())
        if _ds_info["phase"] == "acute":
            _n_dwi = int((_rows["dwi"] == True).sum())
            print(f"{_run_label}: {_n_dwi}/{_n} subjects have DWI on disk, "
                  f"{_n_mask}/{_n} have a lesion mask on disk.")
            if _n_dwi == 0 or _n_mask == 0:
                print("  ⚠ ISLES discovery incomplete — see above for troubleshooting.")
        else:
            _n_t1w = int((_rows["t1w"] == True).sum())
            print(f"{_run_label}: {_n_t1w}/{_n} subjects have T1w on disk, "
                  f"{_n_mask}/{_n} have a lesion mask on disk.")



=== ds004884 — Aphasia Recovery — discovering subjects ===
  sub-M2001/  →  ses-1076, ses-1225, ses-1226, ses-1239, ses-1240, ses-1253
  sub-M2002/  →  ses-1440, ses-1441, ses-1462, ses-1467, ses-1468, ses-1499
  sub-M2003/  →  ses-1408, ses-1503, ses-1505, ses-1555, ses-1559, ses-1573
  … and more
  → 444 subjects with T1w, sampling 2

=== ISLES 2022 — discovering subjects ===
  sub-strokecase0001/  →  .DS_Store, ses-0001
  sub-strokecase0002/  →  ses-0001
  sub-strokecase0003/  →  ses-0001
  … and more
  → 250 subjects with DWI, sampling 2

=== ds006533 — Duricy 2025 (Numeracy/Aphasia) — discovering subjects ===
  (no sub-* directories under /home/jovyan/neurodesktop-storage/calmar/data/ds006533)
  → 0 subjects with T1w, sampling 0

=== ISLES 2015 SISS — discovering subjects ===
  sub-001/  →  anat, dwi
  sub-002/  →  anat, dwi
  sub-003/  →  anat, dwi
  … and more
  → 28 subjects with DWI, sampling 2

=== Synthetic acute stroke (DWI+ADC) — discovering subjects ===
  sub-synacu000/ 

,dataset,subject,session,t1w,dwi,mask,n_masks
0,chronic_ds004884,sub-M2005,ses-2446,True,None,True,1
1,chronic_ds004884,sub-M2156,ses-2653,True,True,True,1
2,acute_isles2022,sub-strokecase0004,ses-0001,None,True,True,1
3,acute_isles2022,sub-strokecase0121,ses-0001,None,True,True,1
4,acute_isles2015,sub-001,,True,True,True,1
5,acute_isles2015,sub-016,,True,True,True,1
6,acute_synthetic_acute,sub-synacu000,,True,True,True,1
7,acute_synthetic_acute,sub-synacu003,,True,True,True,1
8,chronic_synthetic_chronic,sub-synchr000,,True,True,True,1
9,chronic_synthetic_chronic,sub-synchr003,,True,True,True,1


chronic_ds004884: 2/2 subjects have T1w on disk, 2/2 have a lesion mask on disk.
acute_isles2022: 2/2 subjects have DWI on disk, 2/2 have a lesion mask on disk.
acute_isles2015: 2/2 subjects have DWI on disk, 2/2 have a lesion mask on disk.
acute_synthetic_acute: 2/2 subjects have DWI on disk, 2/2 have a lesion mask on disk.
chronic_synthetic_chronic: 2/2 subjects have T1w on disk, 2/2 have a lesion mask on disk.


In [7]:
import inspect
from pathlib import Path

print("running fetch_for has fix:", "_usable" in inspect.getsource(fetch_for))

for info in BENCHMARK_DATASETS.values():
    if info["phase"] != "chronic":
        continue
    print(f"\n=== {info['run_label']} ===")
    for e in info["subjects"]:
        p = e.get("t1w")
        print(f"  {e['subject']} {e['session']}  t1w={p}")
        if p is not None:
            p = Path(p)
            print(f"     exists={p.exists()} symlink={p.is_symlink()} "
                  f"size={p.stat().st_size if p.exists() else 'NA'} is_gzip={is_gzip(p)}")
            if p.is_symlink():
                tgt = p.resolve()
                print(f"     -> {tgt} (exists={tgt.exists()})")

running fetch_for has fix: True

=== chronic_ds004884 ===
  sub-M2005 ses-2446  t1w=/home/jovyan/neurodesktop-storage/calmar/data/ds004884/sub-M2005/ses-2446/anat/sub-M2005_ses-2446_acq-tfl3p2_run-2_T1w.nii.gz
     exists=True symlink=True size=3792239 is_gzip=True
     -> /home/jovyan/neurodesktop-storage/calmar/data/ds004884/.git/annex/objects/zz/Mf/SHA256E-s3792239--41fae888e1f65e33407b05997b0f5f7fb873a7ac1cd0bed29a6fe3e0902e35fd.nii.gz/SHA256E-s3792239--41fae888e1f65e33407b05997b0f5f7fb873a7ac1cd0bed29a6fe3e0902e35fd.nii.gz (exists=True)
  sub-M2156 ses-2653  t1w=/home/jovyan/neurodesktop-storage/calmar/data/ds004884/sub-M2156/ses-2653/anat/sub-M2156_ses-2653_acq-tfl3p2_run-2_T1w.nii.gz
     exists=True symlink=True size=3413827 is_gzip=True
     -> /home/jovyan/neurodesktop-storage/calmar/data/ds004884/.git/annex/objects/vx/8z/SHA256E-s3413827--fa1b528650f79e1822235717829b2bcd8a04969d4304fb9cf23ddff9b51ce77a.nii.gz/SHA256E-s3413827--fa1b528650f79e1822235717829b2bcd8a04969d4304fb9c

## 7 · Reference mask resolver

In [8]:
def reference_mask(entry: dict, dataset_root: Path) -> tuple[Path | None, str]:
    """Locate the dataset's expert lesion mask, auto-fetching annex
    pointers. Tries multiple derivatives layouts (ds004884 vs ISLES)
    and falls back to a different session of the same subject."""
    sub, ses = entry["subject"], entry["session"] or ""

    def _try(p: Path | None, space: str):
        if p is None: return None, "missing"
        got = datalad_get(p) if not is_gzip(p) else p
        return (got, space) if got else (None, "fetch-failed")

    cands = _all_lesion_pointers(dataset_root, sub, ses)
    if not cands:
        return None, "missing"
    # Prefer native-space mask for the active session; fall back to anything
    same_session = [p for p in cands if (not ses) or ses in p.parts]
    pick = same_session[0] if same_session else cands[0]
    if same_session:
        space = "native"
    else:
        # find which session this fallback came from for the label
        ses_parts = [part for part in pick.parts if part.startswith("ses-")]
        space = f"native (alt {ses_parts[0]})" if ses_parts else "native (alt session)"
    return _try(pick, space)


## 9 · Segmentation tools

Each sub-section below defines **and runs** one tool. Cells are independent:
results accumulate in `RUN_LOG` via `_get_or_create_row`, so you can re-run
a single tool cell without touching the others, or skip a broken tool entirely.

To add a new tool: copy any tool cell, update the function and the dataset list,
and append it here — nothing else needs to change.

In [9]:
# ── Shared accumulator — populated by each tool cell below ──────────
import json as _json, shutil as _shutil

RUN_LOG = []
KNOWN_TOOLS = ["linda", "synthstroke", "openads"]

def _get_or_create_row(log, dataset, subject, session, ref_path, ref_space):
    """Find the row for this subject or create one. Tool cells call this
    to upsert their prediction path without overwriting other tools."""
    for row in log:
        if (row["dataset"] == dataset
                and row["subject"] == subject
                and row["session"] == session):
            return row
    row = {
        "dataset": dataset, "subject": subject, "session": session,
        "reference": str(ref_path) if ref_path else "",
        "reference_space": ref_space,
    }
    log.append(row)
    return row

# ── BIDS-derivatives helpers ─────────────────────────────────────────
_BIDS_DATASET_DESC = {
    "linda": {
        "Name": "LINDA",
        "BIDSVersion": "1.9.0",
        "GeneratedBy": [{"Name": "linda", "Version": "0.5.1"}],
    },
    "synthstroke": {
        "Name": "SynthStroke",
        "BIDSVersion": "1.9.0",
        "GeneratedBy": [{"Name": "synthstroke", "Version": "0.0.0"}],
    },
    "openads": {
        "Name": "OpenADS",
        "BIDSVersion": "1.9.0",
        "GeneratedBy": [{"Name": "openadscpu", "Version": "1.0.0"}],
    },
}


def _bids_mask_path(dataset_root, tool: str, sub: str, ses: str,
                    modality: str, space: str) -> Path:
    """Return the canonical BIDS derivatives path for a lesion mask.

    Layout:  derivatives/<tool>/<sub>[/<ses>]/<modality>/
             <sub>[_<ses>]_space-<space>_desc-lesion_mask.nii.gz
    """
    ses_str = ses or ""
    parts = [sub]
    if ses_str:
        parts.append(ses_str)
    fname = "_".join(parts) + f"_space-{space}_desc-lesion_mask.nii.gz"
    if ses_str:
        return Path(dataset_root) / "derivatives" / tool / sub / ses_str / modality / fname
    return Path(dataset_root) / "derivatives" / tool / sub / modality / fname


def _save_bids_mask(src_path, dataset_root, tool: str, sub: str, ses: str,
                    modality: str, space: str) -> Path:
    """Copy *src_path* to its canonical BIDS location; write dataset_description.json."""
    bids = _bids_mask_path(dataset_root, tool, sub, ses, modality, space)
    bids.parent.mkdir(parents=True, exist_ok=True)
    _shutil.copy2(str(src_path), str(bids))
    ds_desc = Path(dataset_root) / "derivatives" / tool / "dataset_description.json"
    if not ds_desc.exists():
        desc = _BIDS_DATASET_DESC.get(tool, {"Name": tool, "BIDSVersion": "1.9.0"})
        ds_desc.write_text(_json.dumps(desc, indent=2))
    return bids


print("RUN_LOG initialised ✔")


RUN_LOG initialised ✔


### 9a · LINDA
T1w-based chronic-stroke segmentation. Runs on **chronic (ds004884)** only — ISLES subjects have no T1w.

In [10]:
import sys as _sys
_sys.path.insert(0, str(PROJECT_DIR))
import linda_qc as _lqc
import importlib as _il
_il.reload(_lqc)


def _preflight_input(p, label: str):
    """Confirm the input image is a materialised gzip NIfTI, not a
    git-annex pointer file. If it isn't, try `datalad_get`. Returns
    True if the file is usable, False otherwise (with an explanatory
    print so the user knows whether to re-run the fetch cell)."""
    if p is None:
        return False
    p = Path(p)
    if not p.exists():
        # Could be an unfetched git-annex pointer (broken symlink -> exists()
        # is False). Try to materialise it before giving up.
        if p.is_symlink():
            got = datalad_get(p)
            if got and got.exists() and is_gzip(got):
                return True
        print(f"  [{label}] preflight: {p} missing on disk")
        return False
    sz = p.stat().st_size
    if not is_gzip(p) or sz < 1024:
        print(f"  [{label}] preflight: {p.name} is {sz} B and not gzip — "
              f"likely an unmaterialised DataLad annex pointer.")
        print(f"           attempting `datalad get` …")
        got = datalad_get(p)
        if not got or not is_gzip(got) or got.stat().st_size < 1024:
            print(f"           ✘ still not materialised. Re-run the fetch")
            print(f"             cell (Section 6) and confirm fetch_summary")
            print(f"             shows the input column True for this subject.")
            return False
        print(f"           ✔ materialised ({got.stat().st_size:,} bytes)")
    return True


def run_linda(entry: dict, dataset_root) -> "Path | None":
    """LINDA on T1w → BIDS derivatives/linda/<sub>/<ses>/anat/<sub>_<ses>_space-T1w_desc-lesion_mask.nii.gz

    Pipeline (mirrors linda-example.ipynb):
      1. HD-BET skull-strip  → derivatives/hdbet/<sub>/<ses>/anat/<stem>_brain_mask.nii.gz
      2. linda_predict_with_mask.sh --t1 … --mask … --outdir …
      3. Copy Prediction3_native.nii.gz to BIDS derivatives path
    """
    sub, ses = entry["subject"], entry["session"] or ""
    t1w = entry.get("t1w")
    if not t1w:
        print(f"  [linda] {sub}/{ses}: no T1w → skip")
        return None
    t1w = Path(t1w)
    if not _preflight_input(t1w, "linda"):
        return None

    # ── BIDS output (canonical cache check) ───────────────────────────
    bids_pred = _bids_mask_path(dataset_root, "linda", sub, ses, "anat", "T1w")
    if CONFIG["SKIP_IF_DONE"] and bids_pred.exists() and is_gzip(bids_pred):
        print(f"  [linda] {sub}/{ses}: cached ✔ (BIDS)")
        return bids_pred

    # ── Working dir (LINDA writes Prediction3_native.nii.gz here) ─────
    work_dir = Path(dataset_root) / "derivatives" / "linda" / sub / ses / "anat"
    work_dir.mkdir(parents=True, exist_ok=True)
    native_pred = work_dir / "Prediction3_native.nii.gz"

    # Also accept Prediction3_native.nii.gz left by linda-example.ipynb
    if CONFIG["SKIP_IF_DONE"] and native_pred.exists() and is_gzip(native_pred):
        print(f"  [linda] {sub}/{ses}: cached ✔ (Prediction3_native) → promoting to BIDS")
        out = _save_bids_mask(native_pred, dataset_root, "linda", sub, ses, "anat", "T1w")
        print(f"  [linda] saved → {out.relative_to(dataset_root)}")
        return out

    # ── Step 1: HD-BET skull-strip ────────────────────────────────────
    # Same dir used by linda-example so the mask is shared between notebooks
    hdbet_dir = Path(dataset_root) / "derivatives" / "hdbet" / sub / ses / "anat"
    hdbet_dir.mkdir(parents=True, exist_ok=True)
    t1_stem = t1w.name
    for ext in (".nii.gz", ".nii"):
        if t1_stem.endswith(ext):
            t1_stem = t1_stem[:-len(ext)]
    mask = hdbet_dir / f"{t1_stem}_brain_mask.nii.gz"

    if mask.exists() and is_gzip(mask):
        print(f"  [linda] {sub}/{ses}: HD-BET mask cached ✔")
    else:
        if _lqc.detect_brain_extractor() is None:
            print(f"  [linda] {sub}/{ses}: no brain extractor on PATH "
                  f"(load hd-bet or fsl/mri_synthstrip module)")
            return None
        print(f"  [linda] {sub}/{ses}: running HD-BET …", end="", flush=True)
        t0 = time.time()
        res = _lqc.run_hd_bet(t1w, hdbet_dir, mode="fast")
        if res.get("returncode") != 0 or not mask.exists():
            print(f" FAILED (rc={res.get('returncode')})")
            return None
        print(f" ✔ ({time.time()-t0:.0f}s)")

    # ── Step 2: LINDA with mask-bypass ────────────────────────────────
    wrapper = PROJECT_DIR / "linda_predict_with_mask.sh"
    if wrapper.exists():
        wrapper.chmod(wrapper.stat().st_mode | 0o111)  # ensure +x
        cmd = [str(wrapper),
               "--t1",     str(t1w),
               "--mask",   str(mask),
               "--outdir", str(work_dir)]
    elif shutil.which("linda_predict_with_mask.sh"):
        cmd = ["linda_predict_with_mask.sh",
               "--t1",     str(t1w),
               "--mask",   str(mask),
               "--outdir", str(work_dir)]
    elif shutil.which("linda_predict.sh"):
        # Fallback: plain LINDA without mask (no HD-BET bypass)
        cmd = ["linda_predict.sh", str(t1w), str(work_dir)]
    else:
        print(f"  [linda] {sub}/{ses}: no LINDA CLI on PATH "
              f"(did you `module.load('linda/...')`?)")
        return None

    print(f"  [linda] {sub}/{ses}: running LINDA (10–30 min) …")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  [linda] {sub}/{ses}: FAILED in {time.time()-t0:.0f}s "
              f"rc={proc.returncode}\n    {proc.stderr.strip()[:2000]}")
        return None
    print(f"  [linda] {sub}/{ses}: done in {time.time()-t0:.0f}s")

    if not native_pred.exists():
        print(f"  [linda] {sub}/{ses}: expected output not found ({native_pred.name})")
        return None

    # ── Step 3: copy to BIDS path ─────────────────────────────────────
    out = _save_bids_mask(native_pred, dataset_root, "linda", sub, ses, "anat", "T1w")
    print(f"  [linda] saved → {out.relative_to(dataset_root)}")
    return out


# LINDA: chronic dataset only
if CONFIG["RUN_LINDA"]:
    # Runs on all active datasets (chronic + acute transfer baseline)
    for label, entries, ds_root in [
        (info["run_label"], info["subjects"], info["root"])
        for info in BENCHMARK_DATASETS.values()
    ]:
        print(f"=== LINDA · {label}: {len(entries)} subject(s) ===")
        for e in entries:
            sub, ses = e["subject"], e["session"] or ""
            print(f"\n• {sub} {ses}")
            ref_path, ref_space = reference_mask(e, ds_root)
            if ref_path is None:
                print(f"  [ref] no expert mask ({ref_space}); metrics will be NaN")
            else:
                print(f"  [ref] {ref_path.name}  ({ref_space})")
            row = _get_or_create_row(RUN_LOG, label, sub, ses, ref_path, ref_space)
            try:
                pred = run_linda(e, ds_root)
                row["linda_pred"] = str(pred) if pred else ""
            except Exception as exc:
                print(f"  [linda] {sub}/{ses}: EXCEPTION {type(exc).__name__}: {exc}")
                row["linda_pred"] = ""
                if CONFIG["STOP_ON_TOOL_ERROR"]:
                    raise
else:
    print("LINDA disabled (CONFIG['RUN_LINDA'] = False)")


=== LINDA · chronic_ds004884: 2 subject(s) ===

• sub-M2005 ses-2446
  [ref] sub-M2005_ses-1034_acq-spc3_run-2_T2w_desc-lesion_mask.nii.gz  (native (alt ses-1034))
  [linda] sub-M2005/ses-2446: cached ✔ (BIDS)

• sub-M2156 ses-2653
  [ref] sub-M2156_ses-2653_acq-spc3_run-3_T2w_desc-lesion_mask.nii.gz  (native)
  [linda] sub-M2156/ses-2653: cached ✔ (BIDS)
=== LINDA · acute_isles2022: 2 subject(s) ===

• sub-strokecase0004 ses-0001
  [ref] sub-strokecase0004_ses-0001_msk.nii.gz  (native)
  [linda] sub-strokecase0004/ses-0001: no T1w → skip

• sub-strokecase0121 ses-0001
  [ref] sub-strokecase0121_ses-0001_msk.nii.gz  (native)
  [linda] sub-strokecase0121/ses-0001: no T1w → skip
=== LINDA · chronic_ds006533: 0 subject(s) ===
=== LINDA · acute_isles2015: 2 subject(s) ===

• sub-001 
  [ref] sub-001_msk.nii.gz  (native)
  [linda] sub-001/: HD-BET mask cached ✔
  [linda] sub-001/: running LINDA (10–30 min) …
  [linda] sub-001/: FAILED in 81s rc=4
    linda_predict() failed:  file does not e

/opt/HD-BET/HD_BET/utils.py:74: UserWarning: The argument 'neighbors' is deprecated and will be removed in scikit-image 0.18, use 'connectivity' instead. For neighbors=8, use connectivity=3
  lbls = label(mask, 8)



########################
If you are using hd-bet, please cite the following paper:
Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W,Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificialneural networks. arXiv preprint arXiv:1901.11341, 2019.
########################

File: /home/jovyan/neurodesktop-storage/calmar/data/synthetic_acute/sub-synacu000/anat/sub-synacu000_T1w.nii.gz
preprocessing...
image shape after preprocessing:  (127, 156, 132)
prediction (CNN id)...
0
running postprocessing... 
exporting segmentation...
 ✔ (14s)
  [linda] sub-synacu000/: running LINDA (10–30 min) …
  [linda] sub-synacu003/: done in 402s
  [linda] saved → derivatives/linda/sub-synacu003/anat/sub-synacu003_space-T1w_desc-lesion_mask.nii.gz
=== LINDA · chronic_synthetic_chronic: 2 subject(s) ===

• sub-synchr000 
  [ref] sub-synchr000_msk.nii.gz  (native)
  [linda] sub-synchr000/: running H

/opt/HD-BET/HD_BET/utils.py:74: UserWarning: The argument 'neighbors' is deprecated and will be removed in scikit-image 0.18, use 'connectivity' instead. For neighbors=8, use connectivity=3
  lbls = label(mask, 8)



########################
If you are using hd-bet, please cite the following paper:
Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W,Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificialneural networks. arXiv preprint arXiv:1901.11341, 2019.
########################

File: /home/jovyan/neurodesktop-storage/calmar/data/synthetic_chronic/sub-synchr000/anat/sub-synchr000_T1w.nii.gz
preprocessing...
image shape after preprocessing:  (127, 156, 132)
prediction (CNN id)...
0
running postprocessing... 
exporting segmentation...
 ✔ (12s)
  [linda] sub-synchr000/: running LINDA (10–30 min) …
  [linda] sub-synchr000/: done in 418s
  [linda] saved → derivatives/linda/sub-synchr000/anat/sub-synchr000_space-T1w_desc-lesion_mask.nii.gz

• sub-synchr003 
  [ref] sub-synchr003_msk.nii.gz  (native)
  [linda] sub-synchr003/: running HD-BET …  ▶ hd-bet -i /home/jovyan/neurodesktop-storage

/opt/HD-BET/HD_BET/utils.py:74: UserWarning: The argument 'neighbors' is deprecated and will be removed in scikit-image 0.18, use 'connectivity' instead. For neighbors=8, use connectivity=3
  lbls = label(mask, 8)



########################
If you are using hd-bet, please cite the following paper:
Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W,Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificialneural networks. arXiv preprint arXiv:1901.11341, 2019.
########################

File: /home/jovyan/neurodesktop-storage/calmar/data/synthetic_chronic/sub-synchr003/anat/sub-synchr003_T1w.nii.gz
preprocessing...
image shape after preprocessing:  (127, 156, 132)
prediction (CNN id)...
0
running postprocessing... 
exporting segmentation...
 ✔ (13s)
  [linda] sub-synchr003/: running LINDA (10–30 min) …
  [linda] sub-synchr003/: done in 404s
  [linda] saved → derivatives/linda/sub-synchr003/anat/sub-synchr003_space-T1w_desc-lesion_mask.nii.gz


### 9b · SynthStroke
Sequence-agnostic segmentation. Runs on **all datasets** (chronic T1w + acute DWI).

In [11]:
def _preflight_input(p, label: str):
    """Confirm the input image is a materialised gzip NIfTI, not a
    git-annex pointer file. If it isn't, try `datalad_get`. Returns
    True if the file is usable, False otherwise (with an explanatory
    print so the user knows whether to re-run the fetch cell)."""
    if p is None:
        return False
    p = Path(p)
    if not p.exists():
        # Could be an unfetched git-annex pointer (broken symlink -> exists()
        # is False). Try to materialise it before giving up.
        if p.is_symlink():
            got = datalad_get(p)
            if got and got.exists() and is_gzip(got):
                return True
        print(f"  [{label}] preflight: {p} missing on disk")
        return False
    sz = p.stat().st_size
    if not is_gzip(p) or sz < 1024:
        print(f"  [{label}] preflight: {p.name} is {sz} B and not gzip — "
              f"likely an unmaterialised DataLad annex pointer.")
        print(f"           attempting `datalad get` …")
        got = datalad_get(p)
        if not got or not is_gzip(got) or got.stat().st_size < 1024:
            print(f"           ✘ still not materialised. Re-run the fetch")
            print(f"             cell (Section 6) and confirm fetch_summary")
            print(f"             shows the input column True for this subject.")
            return False
        print(f"           ✔ materialised ({got.stat().st_size:,} bytes)")
    return True


def _synthstroke_cmd():
    """Auto-detect how SynthStroke is exposed in this environment."""
    for binname in ("synthstroke", "SynthStrokeSPM"):
        if shutil.which(binname):
            return [binname]
    try:
        r = subprocess.run([sys.executable, "-c", "import synthstroke"],
                           capture_output=True)
        if r.returncode == 0:
            return [sys.executable, "-m", "synthstroke"]
    except Exception:
        pass
    return None


def run_synthstroke(entry: dict, dataset_root) -> "Path | None":
    sub, ses = entry["subject"], entry["session"] or ""

    # SynthStroke is T1w-only — all pre-trained models require T1-weighted input.
    # Do NOT fall back to DWI: it produces empty/garbage masks.
    inp = entry.get("t1w")
    if not inp:
        print(f"  [synthstroke] {sub}/{ses}: no T1w → skip (SynthStroke requires T1w)")
        return None

    if not _preflight_input(inp, "synthstroke"):
        return None
    # SynthStroke always uses T1w space
    space, modality = "T1w", "anat"

    # Check BIDS output first (SKIP_IF_DONE)
    bids_pred = _bids_mask_path(dataset_root, "synthstroke", sub, ses, modality, space)
    if CONFIG["SKIP_IF_DONE"] and bids_pred.exists() and is_gzip(bids_pred):
        print(f"  [synthstroke] {sub}/{ses}: cached ✔")
        return bids_pred

    # Working output dir (SynthStroke writes prediction_lesion_mask.nii.gz here)
    work_dir = dataset_root / "derivatives" / "synthstroke" / sub / ses / modality
    work_dir.mkdir(parents=True, exist_ok=True)
    native_pred = work_dir / "prediction_lesion_mask.nii.gz"

    base_cmd = _synthstroke_cmd()
    if base_cmd is None:
        print(f"  [synthstroke] {sub}/{ses}: not on PATH "
              f"(did you `module.load('synthstroke/<ver>')`?)")
        return None

    # CLI: synthstroke --input <file> --outdir <dir>
    cmd = base_cmd + ["--input", str(inp), "--outdir", str(work_dir)]
    print(f"  [synthstroke] {sub}/{ses}: running ({space}) …")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  [synthstroke] {sub}/{ses}: FAILED in {time.time()-t0:.0f}s "
              f"rc={proc.returncode}\n    {(proc.stderr or proc.stdout).strip()[:2000]}")
        return None
    print(f"  [synthstroke] {sub}/{ses}: done in {time.time()-t0:.0f}s")

    if not native_pred.exists():
        print(f"  [synthstroke] {sub}/{ses}: expected output not found ({native_pred.name})")
        return None

    # Copy to BIDS path
    out = _save_bids_mask(native_pred, dataset_root, "synthstroke", sub, ses, modality, space)
    print(f"  [synthstroke] saved → {out.relative_to(dataset_root)}")
    return out


# SynthStroke: all datasets
if CONFIG["RUN_SYNTHSTROKE"]:
    # SynthStroke is T1w-only: auto-skips subjects with no T1w
    for label, entries, ds_root in [
        (info["run_label"], info["subjects"], info["root"])
        for info in BENCHMARK_DATASETS.values()
    ]:
        print(f"=== SynthStroke · {label}: {len(entries)} subject(s) ===")
        for e in entries:
            sub, ses = e["subject"], e["session"] or ""
            print(f"\n• {sub} {ses}")
            ref_path, ref_space = reference_mask(e, ds_root)
            if ref_path is None:
                print(f"  [ref] no expert mask ({ref_space}); metrics will be NaN")
            else:
                print(f"  [ref] {ref_path.name}  ({ref_space})")
            row = _get_or_create_row(RUN_LOG, label, sub, ses, ref_path, ref_space)
            try:
                pred = run_synthstroke(e, ds_root)
                row["synthstroke_pred"] = str(pred) if pred else ""
            except Exception as exc:
                print(f"  [synthstroke] {sub}/{ses}: EXCEPTION {type(exc).__name__}: {exc}")
                row["synthstroke_pred"] = ""
                if CONFIG["STOP_ON_TOOL_ERROR"]:
                    raise
else:
    print("SynthStroke disabled (CONFIG['RUN_SYNTHSTROKE'] = False)")


=== SynthStroke · chronic_ds004884: 2 subject(s) ===

• sub-M2005 ses-2446
  [ref] sub-M2005_ses-1034_acq-spc3_run-2_T2w_desc-lesion_mask.nii.gz  (native (alt ses-1034))
  [synthstroke] sub-M2005/ses-2446: cached ✔

• sub-M2156 ses-2653
  [ref] sub-M2156_ses-2653_acq-spc3_run-3_T2w_desc-lesion_mask.nii.gz  (native)
  [synthstroke] sub-M2156/ses-2653: cached ✔
=== SynthStroke · acute_isles2022: 2 subject(s) ===

• sub-strokecase0004 ses-0001
  [ref] sub-strokecase0004_ses-0001_msk.nii.gz  (native)
  [synthstroke] sub-strokecase0004/ses-0001: no T1w → skip (SynthStroke requires T1w)

• sub-strokecase0121 ses-0001
  [ref] sub-strokecase0121_ses-0001_msk.nii.gz  (native)
  [synthstroke] sub-strokecase0121/ses-0001: no T1w → skip (SynthStroke requires T1w)
=== SynthStroke · chronic_ds006533: 0 subject(s) ===
=== SynthStroke · acute_isles2015: 2 subject(s) ===

• sub-001 
  [ref] sub-001_msk.nii.gz  (native)
  [synthstroke] sub-001/: cached ✔

• sub-016 
  [ref] sub-016_msk.nii.gz  (native)


### 9c · OpenADS
DWI-based acute-stroke segmentation. Runs on **acute (ISLES 2022)** only.

> **Known issue:** `--subject-path` is currently passed at session level
> (`sub-strokecase####/ses-0001`). OpenADS expects the subject-level directory
> and derives the session path internally — fix is to pass `ds_root / sub` instead.

In [12]:
def _preflight_input(p, label: str):
    """Confirm the input image is a materialised gzip NIfTI, not a
    git-annex pointer file. If it isn't, try `datalad_get`. Returns
    True if the file is usable, False otherwise (with an explanatory
    print so the user knows whether to re-run the fetch cell)."""
    if p is None:
        return False
    p = Path(p)
    if not p.exists():
        # Could be an unfetched git-annex pointer (broken symlink -> exists()
        # is False). Try to materialise it before giving up.
        if p.is_symlink():
            got = datalad_get(p)
            if got and got.exists() and is_gzip(got):
                return True
        print(f"  [{label}] preflight: {p} missing on disk")
        return False
    sz = p.stat().st_size
    if not is_gzip(p) or sz < 1024:
        print(f"  [{label}] preflight: {p.name} is {sz} B and not gzip — "
              f"likely an unmaterialised DataLad annex pointer.")
        print(f"           attempting `datalad get` …")
        got = datalad_get(p)
        if not got or not is_gzip(got) or got.stat().st_size < 1024:
            print(f"           ✘ still not materialised. Re-run the fetch")
            print(f"             cell (Section 6) and confirm fetch_summary")
            print(f"             shows the input column True for this subject.")
            return False
        print(f"           ✔ materialised ({got.stat().st_size:,} bytes)")
    return True


def _openads_cmd():
    for binname in ("ads", "openads"):
        if shutil.which(binname):
            return [binname]
    try:
        r = subprocess.run([sys.executable, "-c", "import ads"],
                           capture_output=True)
        if r.returncode == 0:
            return [sys.executable, "-m", "ads.cli"]
    except Exception:
        pass
    return None


def _openads_stage_subject(entry: dict, dataset_root: Path) -> Path | None:
    """Create a flat staging directory with the filenames OpenADS expects.

    OpenADS expects:  sub-xxxx/{id}_DWI.nii.gz  and  {id}_ADC.nii.gz
    where {id} is the subject folder name with the leading "sub-" stripped.
    ISLES puts files at: sub-xxxx/ses-yyyy/dwi/sub-xxxx_ses-yyyy_{dwi|adc}.nii.gz

    Returns the staging subject dir, or None if the source DWI is missing.
    """
    sub, ses = entry["subject"], entry["session"] or ""
    src_dwi = entry.get("dwi")
    if not src_dwi or not Path(src_dwi).exists():
        return None

    # Companion ADC: same folder as DWI, replace _dwi with _adc
    src_adc = Path(str(src_dwi).replace("_dwi.nii.gz", "_adc.nii.gz")
                               .replace("_DWI.nii.gz", "_ADC.nii.gz"))

    # OpenADS strips "sub-" to build filenames: sub-strokecase0086 → strokecase0086
    short_id = sub.removeprefix("sub-")
    staging_dir = dataset_root / "derivatives" / "openads_staging" / sub
    staging_dir.mkdir(parents=True, exist_ok=True)

    dst_dwi = staging_dir / f"{short_id}_DWI.nii.gz"
    dst_adc = staging_dir / f"{short_id}_ADC.nii.gz"

    for dst, src in [(dst_dwi, src_dwi), (dst_adc, src_adc)]:
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        if Path(src).exists():
            dst.symlink_to(Path(src).resolve())
        else:
            print(f"  [openads-stage] warning: {Path(src).name} not found, skipping")

    return staging_dir


def run_openads(entry: dict, dataset_root) -> "Path | None":
    """OpenADS DWI pipeline.

    OpenADS expects a flat subject dir with {id}_DWI.nii.gz / {id}_ADC.nii.gz.
    We create a symlinked staging dir from the BIDS-nested ISLES layout, then
    pass that as --subject-path.
    Output is searched under derivatives/openads/<sub>/DWI/ and then copied to
    the BIDS derivatives path.
    """
    sub, ses = entry["subject"], entry["session"] or ""
    if not entry.get("dwi"):
        print(f"  [openads] {sub}/{ses}: no DWI → skip (OpenADS needs DWI)")
        return None
    if not _preflight_input(entry.get("dwi"), "openads"):
        return None

    # Check BIDS output first (SKIP_IF_DONE)
    bids_pred = _bids_mask_path(dataset_root, "openads", sub, ses, "dwi", "DWI")
    if CONFIG["SKIP_IF_DONE"] and bids_pred.exists() and is_gzip(bids_pred):
        print(f"  [openads] {sub}/{ses}: cached ✔")
        return bids_pred

    out_root = dataset_root / "derivatives" / "openads"
    out_root.mkdir(parents=True, exist_ok=True)

    # OpenADS writes under <out_root>/<sub>/DWI/ — search broadly
    def _find_openads_lesion():
        dwi_root = out_root / sub / "DWI"
        if not dwi_root.exists():
            return None
        for pattern in ["**/lesion_native.nii.gz", "**/lesion*.nii.gz",
                        "**/segment/*lesion*.nii.gz", "**/*lesion*.nii.gz",
                        "**/segment/*stroke-mask.nii.gz", "**/*stroke-mask.nii.gz"]:
            hits = sorted(dwi_root.glob(pattern))
            if hits:
                return hits[-1]
        return None

    base_cmd = _openads_cmd()
    if base_cmd is None:
        print(f"  [openads] {sub}/{ses}: `ads` CLI not on PATH "
              f"(did you `module.load('openadscpu/<ver>')`?)")
        return None

    # Build flat staging dir with filenames OpenADS expects
    staging = _openads_stage_subject(entry, dataset_root)
    if staging is None:
        print(f"  [openads] {sub}/{ses}: could not stage input files")
        return None

    # OpenADS needs a DWI + ADC pair (or a multi-volume DWI with a b0 it can
    # derive ADC from). Datasets like ISLES 2015 SISS ship only a single 3-D
    # DWI trace and no ADC, so OpenADS aborts in preprocessing ("Cannot
    # compute ADC: b0 volume is not available"). Detect that and skip cleanly
    # instead of running the tool to a hard failure.
    _short_id   = sub.removeprefix("sub-")
    _staged_adc = staging / f"{_short_id}_ADC.nii.gz"
    _dwi_3d = False
    try:
        import nibabel as _nib
        _dwi_3d = _nib.load(str(entry["dwi"])).ndim < 4
    except Exception:
        pass
    if not _staged_adc.exists() and _dwi_3d:
        print(f"  [openads] {sub}/{ses}: no ADC map and DWI is single-volume "
              f"(3-D trace) -> OpenADS requires DWI+ADC; skipping "
              f"(this dataset has no ADC).")
        return None

    preprocess_dir = out_root / sub / "DWI" / "preprocess"

    def _run_stages(stages: str) -> bool:
        """Run a set of OpenADS stages silently; print stderr on failure."""
        cmd = base_cmd + ["dwi",
                          "--subject-path", str(staging),
                          "--output-root",  str(out_root),
                          "--stages", stages]
        proc = subprocess.run(cmd, stdout=subprocess.DEVNULL,
                              stderr=subprocess.PIPE, text=True)
        if proc.returncode != 0:
            print(f"\n  [openads] stderr:\n{proc.stderr.strip()[:2000]}")
        return proc.returncode == 0

    print(f"  [openads] {sub}/{ses}: running (pass 1/2) …", end="", flush=True)
    t0 = time.time()
    if not _run_stages("prepdata,gen_mask,skull_strip"):
        print(f" FAILED")
        return None
    print(f" ✔", end="", flush=True)

    # Skull strip saves everything as *_stroke.nii.gz when no stroke reference
    # exists.  Registration expects *_DWI_brain.nii.gz — create a symlink.
    stroke_file = preprocess_dir / f"{sub}_stroke.nii.gz"
    dwi_brain   = preprocess_dir / f"{sub}_DWI_brain.nii.gz"
    adc_brain   = preprocess_dir / f"{sub}_ADC_brain.nii.gz"
    adc_file    = preprocess_dir / f"{sub}_ADC.nii.gz"

    for dst, src in [(dwi_brain, stroke_file), (adc_brain, adc_file)]:
        if src.exists():
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            dst.symlink_to(src.resolve())
            print(f"  [openads] linked {dst.name} → {src.name}")

    print(f"  (pass 2/2) …", end="", flush=True)
    if not _run_stages("registration,inference,report"):
        print(f" FAILED")
        return None

    print(f" ✔  done in {time.time()-t0:.0f}s")
    found = _find_openads_lesion()
    if not found:
        print(f"  [openads] ⚠ no lesion file found under {out_root / sub}/DWI")
        return None

    print(f"  [openads] output → {found.name}")

    # OpenADS reorients input to RAS+ internally but writes the output
    # with the reoriented affine, not the original DWI affine. Correct
    # it before saving so world-space metrics and vis are accurate.
    src_dwi = entry.get("dwi")
    if src_dwi and Path(str(src_dwi)).exists():
        try:
            import nibabel as _nib, numpy as _np
            dwi_img  = _nib.load(str(src_dwi))
            mask_img = _nib.load(str(found))
            # OpenADS reorients its internal grid to RAS+ but does NOT reorder
            # the voxel data array. Just reattach the original DWI affine.
            corrected = _nib.Nifti1Image(
                mask_img.get_fdata().copy(),
                dwi_img.affine, dwi_img.header)
            corrected_path = found.parent / (found.stem + "_affix.nii.gz")
            _nib.save(corrected, str(corrected_path))
            found = corrected_path
            print(f"  [openads] affine corrected (DWI affine attached)")
        except Exception as _e:
            print(f"  [openads] affine correction skipped: {_e}")

    # Copy to BIDS path
    out = _save_bids_mask(found, dataset_root, "openads", sub, ses, "dwi", "DWI")
    print(f"  [openads] saved → {out.relative_to(dataset_root)}")
    return out


# OpenADS: acute (DWI) datasets only — auto-skipped for chronic
if CONFIG["RUN_OPENADS"]:
    for label, entries, ds_root in [
        (info["run_label"], info["subjects"], info["root"])
        for info in BENCHMARK_DATASETS.values()
        if info["phase"] == "acute"
    ]:
        print(f"=== OpenADS · {label}: {len(entries)} subject(s) ===")
        for e in entries:
            sub, ses = e["subject"], e["session"] or ""
            print(f"\n• {sub} {ses}")
            ref_path, ref_space = reference_mask(e, ds_root)
            if ref_path is None:
                print(f"  [ref] no expert mask ({ref_space}); metrics will be NaN")
            else:
                print(f"  [ref] {ref_path.name}  ({ref_space})")
            row = _get_or_create_row(RUN_LOG, label, sub, ses, ref_path, ref_space)
            try:
                pred = run_openads(e, ds_root)
                row["openads_pred"] = str(pred) if pred else ""
            except Exception as exc:
                print(f"  [openads] {sub}/{ses}: EXCEPTION {type(exc).__name__}: {exc}")
                row["openads_pred"] = ""
                if CONFIG["STOP_ON_TOOL_ERROR"]:
                    raise
else:
    print("OpenADS disabled (CONFIG['RUN_OPENADS'] = False)")


=== OpenADS · acute_isles2022: 2 subject(s) ===

• sub-strokecase0004 ses-0001
  [ref] sub-strokecase0004_ses-0001_msk.nii.gz  (native)
  [openads] sub-strokecase0004/ses-0001: cached ✔

• sub-strokecase0121 ses-0001
  [ref] sub-strokecase0121_ses-0001_msk.nii.gz  (native)
  [openads] sub-strokecase0121/ses-0001: cached ✔
=== OpenADS · acute_isles2015: 2 subject(s) ===

• sub-001 
  [ref] sub-001_msk.nii.gz  (native)
  [openads-stage] warning: sub-001_adc.nii.gz not found, skipping
  [openads] sub-001/: no ADC map and DWI is single-volume (3-D trace) -> OpenADS requires DWI+ADC; skipping (this dataset has no ADC).

• sub-016 
  [ref] sub-016_msk.nii.gz  (native)
  [openads-stage] warning: sub-016_adc.nii.gz not found, skipping
  [openads] sub-016/: no ADC map and DWI is single-volume (3-D trace) -> OpenADS requires DWI+ADC; skipping (this dataset has no ADC).
=== OpenADS · acute_synthetic_acute: 2 subject(s) ===

• sub-synacu000 
  [ref] sub-synacu000_msk.nii.gz  (native)
  [openads] s

### Run log
Collect all tool results into a single DataFrame and save to CSV.

In [13]:
run_log_df = pd.DataFrame(RUN_LOG)

# Ensure a column exists for every known tool even if its cell was skipped
for tool in KNOWN_TOOLS:
    col = f"{tool}_pred"
    if col not in run_log_df.columns:
        run_log_df[col] = ""

run_log_df.to_csv(REPORTS_DIR / "run_log.csv", index=False)
print(f"Run log → {REPORTS_DIR / 'run_log.csv'}")
display(run_log_df)


Run log → /home/jovyan/neurodesktop-storage/calmar/reports/benchmarks/run_log.csv


,dataset,subject,session,reference,reference_space,linda_pred,synthstroke_pred,openads_pred
0,chronic_ds004884,sub-M2005,ses-2446,/home/jovyan/neurodesktop-storage/calmar/data/...,native (alt ses-1034),/home/jovyan/neurodesktop-storage/calmar/data/...,/home/jovyan/neurodesktop-storage/calmar/data/...,NaN
1,chronic_ds004884,sub-M2156,ses-2653,/home/jovyan/neurodesktop-storage/calmar/data/...,native,/home/jovyan/neurodesktop-storage/calmar/data/...,/home/jovyan/neurodesktop-storage/calmar/data/...,NaN
2,acute_isles2022,sub-strokecase0004,ses-0001,/home/jovyan/neurodesktop-storage/calmar/data/...,native,,,/home/jovyan/neurodesktop-storage/calmar/data/...
3,acute_isles2022,sub-strokecase0121,ses-0001,/home/jovyan/neurodesktop-storage/calmar/data/...,native,,,/home/jovyan/neurodesktop-storage/calmar/data/...
4,acute_isles2015,sub-001,,/home/jovyan/neurodesktop-storage/calmar/data/...,native,,/home/jovyan/neurodesktop-storage/calmar/data/...,
5,acute_isles2015,sub-016,,/home/jovyan/neurodesktop-storage/calmar/data/...,native,,/home/jovyan/neurodesktop-storage/calmar/data/...,
6,acute_synthetic_acute,sub-synacu000,,/home/jovyan/neurodesktop-storage/calmar/data/...,native,/home/jovyan/neurodesktop-storage/calmar/data/...,/home/jovyan/neurodesktop-storage/calmar/data/...,/home/jovyan/neurodesktop-storage/calmar/data/...
7,acute_synthetic_acute,sub-synacu003,,/home/jovyan/neurodesktop-storage/calmar/data/...,native,/home/jovyan/neurodesktop-storage/calmar/data/...,/home/jovyan/neurodesktop-storage/calmar/data/...,/home/jovyan/neurodesktop-storage/calmar/data/...
8,chronic_synthetic_chronic,sub-synchr000,,/home/jovyan/neurodesktop-storage/calmar/data/...,native,/home/jovyan/neurodesktop-storage/calmar/data/...,/home/jovyan/neurodesktop-storage/calmar/data/...,NaN
9,chronic_synthetic_chronic,sub-synchr003,,/home/jovyan/neurodesktop-storage/calmar/data/...,native,/home/jovyan/neurodesktop-storage/calmar/data/...,/home/jovyan/neurodesktop-storage/calmar/data/...,NaN


## 10 · Metrics — Dice / IoU / volume / surface distance

In [14]:
def compute_metrics(ref_path: str, pred_path: str) -> dict:
    out = {"dice": np.nan, "iou": np.nan,
           "ref_vol_ml": np.nan, "pred_vol_ml": np.nan,
           "vol_diff_ml": np.nan, "vol_diff_pct": np.nan,
           "hd95_mm": np.nan, "mean_surf_mm": np.nan,
           "ref_voxels": np.nan, "pred_voxels": np.nan, "error": ""}
    if not ref_path or not pred_path:
        out["error"] = "missing path"; return out
    try:
        ref_img  = nib.load(ref_path)
        pred_img = nib.load(pred_path)
    except Exception as e:
        out["error"] = f"load: {type(e).__name__}: {e}"; return out
    try:
        pred_rs = resample_to_img(pred_img, ref_img, interpolation="nearest")
    except Exception as e:
        out["error"] = f"resample: {type(e).__name__}: {e}"; return out

    ref_arr  = binarise(ref_img); pred_arr = binarise(pred_rs)
    out["ref_voxels"]  = int(ref_arr.sum())
    out["pred_voxels"] = int(pred_arr.sum())
    out["dice"], out["iou"] = dice_iou(ref_arr, pred_arr)

    vv = voxel_volume_mm3(ref_img)
    out["ref_vol_ml"]   = volume_ml(ref_arr,  vv)
    out["pred_vol_ml"]  = volume_ml(pred_arr, vv)
    out["vol_diff_ml"]  = out["pred_vol_ml"] - out["ref_vol_ml"]
    out["vol_diff_pct"] = (100 * out["vol_diff_ml"] / out["ref_vol_ml"]
                           if out["ref_vol_ml"] else np.nan)
    if CONFIG["HD95"]:
        out.update(surface_distances(ref_arr, pred_arr,
                                     ref_img.header.get_zooms()[:3]))
    return out


metric_rows = []
for row in RUN_LOG:
    for tool in KNOWN_TOOLS:
        pred = row.get(f"{tool}_pred", "")
        if not pred: continue
        m = compute_metrics(row["reference"], pred)
        phase = "acute" if row["dataset"].startswith("acute") else "chronic"
        m.update({"dataset": row["dataset"], "phase": phase,
                  "subject": row["subject"],
                  "session": row["session"], "tool": tool})
        metric_rows.append(m)

METRICS = pd.DataFrame(metric_rows)
if not METRICS.empty:
    cols = ["dataset","phase","subject","session","tool","dice","iou",
            "vol_diff_ml","vol_diff_pct","hd95_mm","mean_surf_mm",
            "ref_vol_ml","pred_vol_ml","ref_voxels","pred_voxels","error"]
    METRICS = METRICS[[c for c in cols if c in METRICS.columns]]
    METRICS.to_csv(REPORTS_DIR / "per_subject_metrics.csv", index=False)
    print(f"Per-subject metrics → {REPORTS_DIR / 'per_subject_metrics.csv'}")
    display(METRICS.round(3))
else:
    print("No metrics — no (ref, pred) pair succeeded.")


Per-subject metrics → /home/jovyan/neurodesktop-storage/calmar/reports/benchmarks/per_subject_metrics.csv


,dataset,phase,subject,session,tool,dice,iou,vol_diff_ml,vol_diff_pct,hd95_mm,mean_surf_mm,ref_vol_ml,pred_vol_ml,ref_voxels,pred_voxels,error
0,chronic_ds004884,chronic,sub-M2005,ses-2446,linda,0.389,0.242,109.598,232.549,32.031,10.269,47.129,156.727,47129,156727,
1,chronic_ds004884,chronic,sub-M2005,ses-2446,synthstroke,0.379,0.234,-9.048,-19.198,14.192,5.056,47.129,38.081,47129,38081,
2,chronic_ds004884,chronic,sub-M2156,ses-2653,linda,0.404,0.253,82.934,246.688,45.100,12.579,33.619,116.554,33619,116553,
3,chronic_ds004884,chronic,sub-M2156,ses-2653,synthstroke,0.695,0.532,4.087,12.157,8.602,2.785,33.619,37.706,33619,37706,
4,acute_isles2022,acute,sub-strokecase0004,ses-0001,openads,0.011,0.006,18.112,805.694,69.109,31.062,2.248,20.360,281,2545,
5,acute_isles2022,acute,sub-strokecase0121,ses-0001,openads,0.001,0.000,25.752,628.711,87.593,49.569,4.096,29.848,512,3731,
6,acute_isles2015,acute,sub-001,,synthstroke,0.572,0.401,-83.453,-59.331,13.038,5.620,140.657,57.204,140657,57204,
7,acute_isles2015,acute,sub-016,,synthstroke,0.000,0.000,-0.383,-100.000,NaN,NaN,0.383,0.000,383,0,
8,acute_synthetic_acute,acute,sub-synacu000,,linda,0.341,0.206,47.792,253.136,66.615,15.985,18.880,66.672,2360,8334,
9,acute_synthetic_acute,acute,sub-synacu000,,synthstroke,0.000,0.000,-18.768,-99.407,77.981,61.328,18.880,0.112,2360,14,


## 11 · HarvardOxford per-region overlap

In [15]:
from nilearn.datasets import fetch_atlas_harvard_oxford

def atlas_region_overlap(mask_path: str, atlas_img, atlas_labels) -> pd.DataFrame:
    if not mask_path:
        return pd.DataFrame()
    try:
        m_img = nib.load(mask_path)
    except Exception:
        return pd.DataFrame()
    m_rs = resample_to_img(m_img, atlas_img, interpolation="nearest")
    m    = binarise(m_rs)
    a    = atlas_img.get_fdata().astype(np.uint8)
    tot  = int(m.sum())
    if tot == 0:
        return pd.DataFrame()
    rows = []
    for idx, label in enumerate(atlas_labels):
        if idx == 0:  # background
            continue
        overlap = int(((a == idx) & (m == 1)).sum())
        if overlap >= 3:
            rows.append({"region": label, "overlap_voxels": overlap,
                         "overlap_pct": round(100 * overlap / tot, 2)})
    rows.sort(key=lambda r: -r["overlap_voxels"])
    return pd.DataFrame(rows)


print("Fetching HarvardOxford atlas (cached after first run) …")
ho = fetch_atlas_harvard_oxford(CONFIG["ATLAS"])
ATLAS_IMG, ATLAS_LABELS = ho.maps, ho.labels

region_rows = []
for row in RUN_LOG:
    sources = [("reference", row["reference"])] + [
        (tool, row.get(f"{tool}_pred", "")) for tool in KNOWN_TOOLS
    ]
    for which, path in sources:
        df = atlas_region_overlap(path, ATLAS_IMG, ATLAS_LABELS)
        if df.empty: continue
        df["dataset"] = row["dataset"]
        df["subject"] = row["subject"]
        df["session"] = row["session"]
        df["source"]  = which
        region_rows.append(df)

REGIONS = pd.concat(region_rows, ignore_index=True) if region_rows else pd.DataFrame()
if not REGIONS.empty:
    REGIONS.to_csv(REPORTS_DIR / "region_overlap.csv", index=False)
    print(f"Region overlap → {REPORTS_DIR / 'region_overlap.csv'}  "
          f"({len(REGIONS)} rows)")
    display(REGIONS.head(20))
else:
    print("No region overlaps to report.")


Fetching HarvardOxford atlas (cached after first run) …


[fetch_atlas_harvard_oxford] Dataset found in /home/jovyan/nilearn_data/fsl

Region overlap → /home/jovyan/neurodesktop-storage/calmar/reports/benchmarks/region_overlap.csv  (171 rows)


,region,overlap_voxels,overlap_pct,dataset,subject,session,source
0,Middle Frontal Gyrus,261,4.39,chronic_ds004884,sub-M2005,ses-2446,reference
1,Insular Cortex,234,3.94,chronic_ds004884,sub-M2005,ses-2446,reference
2,Frontal Pole,191,3.21,chronic_ds004884,sub-M2005,ses-2446,reference
3,Frontal Opercular Cortex,191,3.21,chronic_ds004884,sub-M2005,ses-2446,reference
4,"Inferior Frontal Gyrus, pars triangularis",88,1.48,chronic_ds004884,sub-M2005,ses-2446,reference
5,"Inferior Frontal Gyrus, pars opercularis",69,1.16,chronic_ds004884,sub-M2005,ses-2446,reference
6,Superior Frontal Gyrus,31,0.52,chronic_ds004884,sub-M2005,ses-2446,reference
7,Paracingulate Gyrus,31,0.52,chronic_ds004884,sub-M2005,ses-2446,reference
8,Precentral Gyrus,30,0.50,chronic_ds004884,sub-M2005,ses-2446,reference
9,Frontal Orbital Cortex,26,0.44,chronic_ds004884,sub-M2005,ses-2446,reference


## 12 · Aggregate report

In [16]:
if METRICS.empty:
    display(HTML("<p style='color:#c00'>No metrics to aggregate.</p>"))
else:
    # ── Add phase column if not already present ──────────────────────
    if "phase" not in METRICS.columns:
        METRICS["phase"] = METRICS["dataset"].apply(
            lambda d: "acute" if d.startswith("acute") else "chronic")

    # ── Per-subject table (grouped by phase then dataset) ────────────
    METRICS.to_csv(REPORTS_DIR / "per_subject_metrics.csv", index=False)

    # ── Descriptive summary: phase × tool ────────────────────────────
    def _agg_col(col):
        return {
            f"{col}_mean":   (col, "mean"),
            f"{col}_std":    (col, "std"),
            f"{col}_median": (col, "median"),
            f"{col}_min":    (col, "min"),
            f"{col}_max":    (col, "max"),
        }

    agg_spec = {"n": ("dice", "count")}
    for c in ["dice", "iou", "vol_diff_ml", "hd95_mm"]:
        if c in METRICS.columns:
            agg_spec.update(_agg_col(c))

    summary_phase = (METRICS.groupby(["phase", "tool"])
                             .agg(**agg_spec)
                             .round(3).reset_index())
    summary_phase.to_csv(REPORTS_DIR / "summary_by_phase_tool.csv", index=False)

    # Also keep the old dataset-level summary for detail
    summary_ds = (METRICS.groupby(["dataset", "tool"])
                          .agg(n=("dice", "count"),
                               dice_mean=("dice", "mean"),
                               dice_std=("dice", "std"),
                               dice_median=("dice", "median"),
                               iou_mean=("iou", "mean"),
                               vol_diff_ml_mean=("vol_diff_ml", "mean"),
                               hd95_mean=("hd95_mm", "mean"))
                          .round(3).reset_index())
    summary_ds.to_csv(REPORTS_DIR / "summary_by_dataset_tool.csv", index=False)

    print(f"Summary (phase × tool) → {REPORTS_DIR / 'summary_by_phase_tool.csv'}")

    # ── HTML report ───────────────────────────────────────────────────
    CSS = """<style>
    .bench { border-collapse:collapse; font-size:0.87em; margin:8px 0 16px }
    .bench th { background:#2c3e50; color:#fff; padding:6px 10px; text-align:left }
    .bench td { padding:4px 10px; border-bottom:1px solid #eee }
    .bench tr:hover td { background:#f7f7f7 }
    .phase-head { margin:18px 0 4px; padding:6px 12px;
                  background:#1a6b8a; color:#fff; border-radius:3px; display:inline-block }
    .ds-head { margin:12px 0 2px; color:#444; font-size:0.9em }
    </style>"""

    html = [CSS, '<div style="font-family:sans-serif;max-width:1300px">']
    html.append("<h2>Lesion-segmentation benchmark</h2>")
    _ds_desc = ", ".join(
        f"{info['label']} ({len(info['subjects'])} subj)"
        for info in BENCHMARK_DATASETS.values()
    )
    html.append(f"<p style='color:#555'>Datasets: {_ds_desc}. "
                f"Atlas: {CONFIG['ATLAS']}. Predictions resampled (nearest-neighbour) to reference grid.</p>")

    # ── Summary table: phase × tool ──────────────────────────────────
    html.append("<h3>Summary by phase &times; tool</h3>")

    # Build a nicely formatted summary for display
    disp_cols = ["phase", "tool", "n"]
    for c in ["dice", "iou", "vol_diff_ml", "hd95_mm"]:
        mean_c = f"{c}_mean"
        std_c  = f"{c}_std"
        med_c  = f"{c}_median"
        if mean_c in summary_phase.columns:
            disp_cols += [mean_c, std_c, med_c]
    disp_summary = summary_phase[[c for c in disp_cols if c in summary_phase.columns]]
    html.append(disp_summary.to_html(index=False, classes="bench", border=0))

    # ── Per-phase / per-dataset breakdown ────────────────────────────
    for phase in sorted(METRICS["phase"].unique()):
        html.append(f"<div class='phase-head'>Phase: {phase.upper()}</div>")
        phase_df = METRICS[METRICS["phase"] == phase]
        for ds in sorted(phase_df["dataset"].unique()):
            html.append(f"<p class='ds-head'><b>Dataset:</b> {ds}</p>")
            ds_df = (phase_df[phase_df["dataset"] == ds]
                     .drop(columns=[c for c in ["error","phase"] if c in phase_df.columns])
                     .round(3))
            html.append(ds_df.to_html(index=False, classes="bench", border=0))

    html.append("</div>")
    report_html = "\n".join(html)
    report_path = REPORTS_DIR / "benchmark_report.html"
    report_path.write_text(report_html)
    print(f"HTML report → {report_path}")

    # ── Display in notebook ───────────────────────────────────────────
    print("\n── Summary: phase × tool ──")
    display(disp_summary)

    print("\n── Per-subject metrics by phase ──")
    for phase in sorted(METRICS["phase"].unique()):
        print(f"\n  ▸ {phase.upper()}")
        display(METRICS[METRICS["phase"] == phase]
                .drop(columns=[c for c in ["error"] if c in METRICS.columns])
                .round(3))


Summary (phase × tool) → /home/jovyan/neurodesktop-storage/calmar/reports/benchmarks/summary_by_phase_tool.csv
HTML report → /home/jovyan/neurodesktop-storage/calmar/reports/benchmarks/benchmark_report.html

── Summary: phase × tool ──


,phase,tool,n,dice_mean,dice_std,dice_median,iou_mean,iou_std,iou_median,vol_diff_ml_mean,vol_diff_ml_std,vol_diff_ml_median,hd95_mm_mean,hd95_mm_std,hd95_mm_median
0,acute,linda,2,0.171,0.241,0.171,0.103,0.146,0.103,34.460,18.854,34.460,70.416,5.374,70.416
1,acute,openads,4,0.469,0.535,0.462,0.439,0.504,0.423,11.696,12.226,9.908,40.175,44.723,35.554
2,acute,synthstroke,4,0.143,0.286,0.000,0.100,0.201,0.000,-29.805,36.695,-17.692,45.510,45.921,45.510
3,chronic,linda,4,0.204,0.223,0.205,0.126,0.140,0.126,63.235,41.513,64.071,61.965,27.588,64.250
4,chronic,synthstroke,4,0.268,0.336,0.190,0.192,0.252,0.117,-7.252,8.939,-7.744,24.694,23.201,14.192



── Per-subject metrics by phase ──

  ▸ ACUTE


,dataset,phase,subject,session,tool,dice,iou,vol_diff_ml,vol_diff_pct,hd95_mm,mean_surf_mm,ref_vol_ml,pred_vol_ml,ref_voxels,pred_voxels
4,acute_isles2022,acute,sub-strokecase0004,ses-0001,openads,0.011,0.006,18.112,805.694,69.109,31.062,2.248,20.360,281,2545
5,acute_isles2022,acute,sub-strokecase0121,ses-0001,openads,0.001,0.000,25.752,628.711,87.593,49.569,4.096,29.848,512,3731
6,acute_isles2015,acute,sub-001,,synthstroke,0.572,0.401,-83.453,-59.331,13.038,5.620,140.657,57.204,140657,57204
7,acute_isles2015,acute,sub-016,,synthstroke,0.000,0.000,-0.383,-100.000,NaN,NaN,0.383,0.000,383,0
8,acute_synthetic_acute,acute,sub-synacu000,,linda,0.341,0.206,47.792,253.136,66.615,15.985,18.880,66.672,2360,8334
9,acute_synthetic_acute,acute,sub-synacu000,,synthstroke,0.000,0.000,-18.768,-99.407,77.981,61.328,18.880,0.112,2360,14
10,acute_synthetic_acute,acute,sub-synacu000,,openads,0.951,0.907,1.216,6.441,2.000,0.457,18.880,20.096,2360,2512
11,acute_synthetic_acute,acute,sub-synacu003,,linda,0.000,0.000,21.128,127.155,74.216,48.956,16.616,37.744,2077,4718
12,acute_synthetic_acute,acute,sub-synacu003,,synthstroke,0.000,0.000,-16.616,-100.000,NaN,NaN,16.616,0.000,2077,0
13,acute_synthetic_acute,acute,sub-synacu003,,openads,0.914,0.841,1.704,10.255,2.000,0.664,16.616,18.320,2077,2290



  ▸ CHRONIC


,dataset,phase,subject,session,tool,dice,iou,vol_diff_ml,vol_diff_pct,hd95_mm,mean_surf_mm,ref_vol_ml,pred_vol_ml,ref_voxels,pred_voxels
0,chronic_ds004884,chronic,sub-M2005,ses-2446,linda,0.389,0.242,109.598,232.549,32.031,10.269,47.129,156.727,47129,156727
1,chronic_ds004884,chronic,sub-M2005,ses-2446,synthstroke,0.379,0.234,-9.048,-19.198,14.192,5.056,47.129,38.081,47129,38081
2,chronic_ds004884,chronic,sub-M2156,ses-2653,linda,0.404,0.253,82.934,246.688,45.100,12.579,33.619,116.554,33619,116553
3,chronic_ds004884,chronic,sub-M2156,ses-2653,synthstroke,0.695,0.532,4.087,12.157,8.602,2.785,33.619,37.706,33619,37706
14,chronic_synthetic_chronic,chronic,sub-synchr000,,linda,0.000,0.000,45.208,701.988,83.400,40.758,6.440,51.648,805,6456
15,chronic_synthetic_chronic,chronic,sub-synchr000,,synthstroke,0.000,0.000,-6.440,-100.000,NaN,NaN,6.440,0.000,805,0
16,chronic_synthetic_chronic,chronic,sub-synchr003,,linda,0.021,0.011,15.200,85.393,87.330,52.309,17.800,33.000,2225,4125
17,chronic_synthetic_chronic,chronic,sub-synchr003,,synthstroke,0.000,0.000,-17.608,-98.921,51.289,37.458,17.800,0.192,2225,24


## 13 · Lesion mask visualisation

Interactive NiiVue viewer — select a subject from the dropdown to see all available lesion masks overlaid on the T1w (chronic) or DWI (acute) reference image.  Each tool gets a distinct colour; the ground-truth mask is shown in green.


In [17]:
import tempfile, warnings
import nibabel as nib
from nilearn.image import resample_to_img
from ipyniivue import NiiVue
import ipywidgets as widgets
from IPython.display import display, HTML

# Tool overlay colours
TOOL_COLORS = {
    "reference":   "green",
    "linda":       "red",
    "synthstroke": "blue",
    "openads":     "yellow",
}
TOOL_LABELS = {
    "reference":   "Ground truth",
    "linda":       "LINDA",
    "synthstroke": "SynthStroke",
    "openads":     "OpenADS",
}

# Temp dir for resampled masks (cleared on kernel restart)
_VIS_TMP = Path(tempfile.mkdtemp(prefix="lesion_vis_"))


def _resample_to_ref(mask_path, ref_path):
    """Resample binary mask to ref voxel grid (nearest-neighbour).
    Saves resampled file to _VIS_TMP so NiiVue can load it.
    Returns path to resampled file, or original if grids already match.
    """
    try:
        ref_img  = nib.load(str(ref_path))
        mask_img = nib.load(str(mask_path))
        if (mask_img.shape == ref_img.shape and
                abs(mask_img.affine - ref_img.affine).max() < 1e-3):
            return mask_path
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rs = resample_to_img(mask_img, ref_img, interpolation="nearest")
        # Strip both .nii and .gz so we always produce a .nii.gz filename
        stem = Path(mask_path).name
        for ext in (".nii.gz", ".nii"):
            if stem.endswith(ext):
                stem = stem[:-len(ext)]; break
        out = _VIS_TMP / f"{stem}_rs.nii.gz"
        nib.save(rs, str(out))
        return out
    except Exception as exc:
        print(f"    [vis] resample warning: {exc}")
        return mask_path


# Build subject list from RUN_LOG
_subject_options = []
for row in RUN_LOG:
    sub = row.get("subject", "")
    ses = row.get("session", "") or ""
    label = f"{sub}  {ses}".strip()
    _subject_options.append((label, row))

if not _subject_options:
    print("RUN_LOG is empty - run the segmentation cells first.")
else:
    def _ref_volume(row):
        sub = row["subject"]
        # Search all active datasets for this subject
        all_entries = [
            e for info in BENCHMARK_DATASETS.values()
            for e in info["subjects"]
        ]
        entry = next((e for e in all_entries if e["subject"] == sub), None)
        if entry is None:
            return None
        ref = entry.get("t1w") or entry.get("dwi")
        return Path(str(ref)) if ref and Path(str(ref)).exists() else None

    def _build_viewer(row):
        bg_path = _ref_volume(row)
        volumes = []
        if bg_path:
            volumes.append({"path": str(bg_path)})
        else:
            ref = row.get("reference")
            if ref and Path(ref).exists():
                bg_path = Path(ref)
                volumes.append({"path": str(bg_path)})

        if bg_path is None:
            return HTML("<p style='color:#888'>No background image found.</p>")

        # Overlays - resample each mask to background grid so all masks align
        for key in ["reference"] + KNOWN_TOOLS:
            pred_key = "reference" if key == "reference" else f"{key}_pred"
            path_str = row.get(pred_key, "")
            if not path_str:
                continue
            p = Path(path_str)
            if not p.exists():
                continue
            p_rs = _resample_to_ref(p, bg_path)
            volumes.append({"path": str(p_rs), "colormap": TOOL_COLORS.get(key, "red"), "opacity": 0.55})

        if len(volumes) <= 1:
            return HTML("<p style='color:#888'>No masks available yet for this subject.</p>")

        nv = NiiVue(height=380)
        nv.load_volumes(volumes)
        return nv

    # Colour legend
    legend_html = "<div style='display:flex;gap:18px;padding:6px 0 10px'>"
    for key, color in TOOL_COLORS.items():
        legend_html += (
            f"<span><span style='display:inline-block;width:14px;height:14px;"
            f"background:{color};opacity:0.8;border-radius:2px;vertical-align:middle;"
            f"margin-right:4px'></span>{TOOL_LABELS[key]}</span>"
        )
    legend_html += "</div>"

    # Dropdown + viewer
    dropdown = widgets.Dropdown(
        options=[(lbl, i) for i, (lbl, _) in enumerate(_subject_options)],
        description="Subject:",
        layout=widgets.Layout(width="380px"),
    )
    viewer_out = widgets.Output()

    def _refresh(idx):
        _, row = _subject_options[idx]
        viewer_out.clear_output(wait=True)
        with viewer_out:
            display(_build_viewer(row))

    def _on_change(change):
        if change["name"] == "value":
            _refresh(change["new"])

    dropdown.observe(_on_change, names="value")

    display(HTML(legend_html))
    display(dropdown)
    display(viewer_out)
    _refresh(0)


Dropdown(description='Subject:', layout=Layout(width='380px'), options=(('sub-M2005  ses-2446', 0), ('sub-M215…

Output()

## Notes

- `module.load(...)` is async — Jupyter accepts top-level `await` thanks to IPython's autoawait. If running outside Neurodesk, the cell prints a warning and continues, and the runners just say "CLI not on PATH" when they hit that tool.
- `datalad install` is metadata-only (small). Actual NIfTI payload is fetched only for the subjects you sample.
- For chronic strokes the expert mask is in T2w-native space and LINDA's prediction is in T1w-native space. Resampling one to the other's grid is a rough proxy — for publication numbers you'd want ANTs to coregister T1↔T2 first.
- `CONFIG["SUBJECT_FILTER"] = ["sub-M2044", "sub-M2120"]` will force the benchmark onto specific subjects regardless of sampling.


## References

### Tools benchmarked

**LINDA** (Lesion Identification with Neighborhood Data Analysis)  
Pustina D, Coslett HB, Turkeltaub PE, Tustison N, Schwartz MF, Avants B (2016).  
Automated segmentation of chronic stroke lesions using LINDA: Lesion Identification with Neighborhood Data Analysis.  
*Human Brain Mapping*, 37(4), 1405–1421. https://doi.org/10.1002/hbm.23240

**SynthStroke**  
Refer to the [SynthStroke Neurodesk documentation](https://www.neurodesk.org/) for the current version citation.

**OpenADS**  
Ito KL, Kumar A, Zavaliangos-Petropulu A, Cramer SC, Liew SL (2019).  
Pipeline for Analyzing Lesions After Stroke (PALS).  
*Frontiers in Neuroinformatics*, 13, 63. https://doi.org/10.3389/fninf.2019.00063  
*(OpenADS is the open-source successor; cite the PALS paper plus the [OpenADS GitHub](https://github.com/npnl/OpenADS) if used.)*

### Datasets

**ds004884** (BIDS OpenNeuro chronic aphasia dataset)  
Cite the dataset DOI as listed on OpenNeuro: https://openneuro.org/datasets/ds004884

**ISLES 2022** (acute ischaemic stroke)  
Hernandez Petzsche MR, de la Rosa E, Hanning U, Wiest R, Valenzuela W, Reyes M, ... Kirschke JS (2022).  
ISLES 2022: A multi-center magnetic resonance imaging stroke lesion segmentation dataset.  
*Scientific Data*, 9, 762. https://doi.org/10.1038/s41597-022-01875-5

### Metrics & evaluation

**Dice similarity coefficient**: Dice LR (1945). *Ecology*, 26(3), 297–302. https://doi.org/10.2307/1932409  
**Surface Hausdorff distance**: Huttenlocher DP, Klanderman GA, Rucklidge WJ (1993). *IEEE TPAMI*, 15(9), 850–863. https://doi.org/10.1109/34.232073

### Atlas

**Harvard–Oxford cortical atlas** (used for regional overlap analysis)  
Makris N et al. (2006). *Cerebral Cortex*, 16(9), 1231–1242. https://doi.org/10.1093/cercor/bhj100

### Platform

**Neurodesk**  
Renton AI et al. (2024). Neurodesk: an accessible, flexible and portable data analysis environment for reproducible neuroimaging.  
*Nature Methods*, 21, 804–808. https://doi.org/10.1038/s41592-023-02145-x

**nilearn**  
Abraham A et al. (2014). Machine learning for neuroimaging with scikit-learn.  
*Frontiers in Neuroinformatics*, 8, 14. https://doi.org/10.3389/fninf.2014.00014
